In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ==================================================================================================
# 12_development_oof_probability_calibration_audit.ipynb
# COMPLETE SINGLE-CELL NOTEBOOK
# ==================================================================================================
#
# PURPOSE
# -------
# Post hoc, development-only probability-calibration audit for the selected
# Swin and ViT model-loss systems.
#
# PRIMARY INPUTS
# --------------
# Saved development out-of-fold image probabilities from Notebook 06:
#   - selected Swin system
#   - selected ViT system
#   - seeds 42, 2025, and 2026
#   - all five held-out folds pooled within each seed
#
# CALIBRATION METRICS
# -------------------
# 1. Multiclass negative log-likelihood
# 2. Multiclass Brier score
# 3. Ordinal ranked probability score
# 4. Top-label expected calibration error
# 5. Top-label maximum calibration error
# 6. Top-label root-mean-square calibration error
# 7. Macro classwise one-vs-rest ECE
# 8. Per-class one-vs-rest ECE
# 9. Mean confidence and signed confidence-minus-accuracy gap
#
# CALIBRATION LEVELS
# ------------------
# 1. Image level:
#       OOF image probabilities versus released image-level MES labels
#
# 2. Patient level:
#       mean-probability patient predictions versus modal-low derived reference
#
# SUMMARIES
# ---------
# A. Calibration metrics for every system and seed
# B. Mean ± sample SD across seeds
# C. Three-seed OOF probability ensemble, for descriptive reliability diagrams
# D. Patient-cluster stratified bootstrap CIs for the seed-ensemble metrics
# E. Paired Swin-minus-ViT bootstrap differences
# F. ECE sensitivity using 10, 15, and 20 equal-width bins
#
# IMPORTANT GUARDRAILS
# --------------------
# This notebook:
#   - performs NO training
#   - loads NO model checkpoint
#   - accesses NO raw image
#   - accesses NO official-test or official-partition prediction
#   - fits NO calibration model
#   - performs NO temperature scaling
#   - performs NO isotonic regression
#   - optimizes NO threshold
#   - performs NO model or aggregation reselection
#   - changes NO locked result
#
# ==================================================================================================


# ==================================================================================================
# 0. Install missing lightweight dependencies
# ==================================================================================================

import sys
import subprocess
import importlib.util


REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
}


for import_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package_name,
            ]
        )


# ==================================================================================================
# 1. Imports
# ==================================================================================================

import os
import re
import json
import math
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.metrics import cohen_kappa_score

from IPython.display import display


warnings.filterwarnings("ignore")


# ==================================================================================================
# 2. Mount Google Drive
# ==================================================================================================

try:
    from google.colab import drive

    (drive.mount(
        "/content/drive",
        force_remount=False,
    ) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)

except Exception as mount_error:
    print("Google Drive was not mounted automatically.")
    print("Reason:", repr(mount_error))


# ==================================================================================================
# 3. Fixed analysis settings
# ==================================================================================================

NUM_CLASSES = 4

CLASS_IDS = [
    0,
    1,
    2,
    3,
]

CLASS_NAMES = {
    0: "Mayo 0",
    1: "Mayo 1",
    2: "Mayo 2",
    3: "Mayo 3",
}

PROBABILITY_COLUMNS = [
    "prob_0",
    "prob_1",
    "prob_2",
    "prob_3",
]

CLASS_VALUES = np.asarray(
    CLASS_IDS,
    dtype=float,
)


TRAINING_SEEDS = [
    42,
    2025,
    2026,
]


EXPECTED_DEVELOPMENT_IMAGES = 9590
EXPECTED_DEVELOPMENT_PATIENTS = 478

EXPECTED_MODAL_LOW_SUPPORT = {
    0: 293,
    1: 108,
    2: 46,
    3: 31,
}


# These values identify the exact Notebook 06 OOF sources.
EXPECTED_PATIENT_QWK = {
    ("Swin", 42): 0.866978,
    ("Swin", 2025): 0.877706,
    ("Swin", 2026): 0.885770,
    ("ViT", 42): 0.842837,
    ("ViT", 2025): 0.839909,
    ("ViT", 2026): 0.865410,
}

SOURCE_QWK_TOLERANCE = 0.010


# Calibration specification
PRIMARY_ECE_BINS = 15

ECE_BIN_SENSITIVITY = [
    10,
    15,
    20,
]

PROBABILITY_EPSILON = 1e-12


# Bootstrap specification
BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = 20260901


# Figure specification
PNG_DPI = 600

SAVE_PNG = True
SAVE_SVG = True
SAVE_PDF = True

SHOW_FIGURES = True


# Colorblind-friendly colors
SWIN_COLOR = "#0072B2"
VIT_COLOR = "#D55E00"
REFERENCE_COLOR = "#222222"
NEUTRAL_COLOR = "#777777"
LIGHT_GRAY = "#D9D9D9"


mpl.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.titlesize": 14,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

sns.set_theme(
    style="whitegrid",
    context="paper",
)


# Optional manual file overrides.
#
# Leave every value as None for automatic discovery.
# Enter an exact CSV path only when automatic discovery fails.
DEVELOPMENT_FILE_OVERRIDES = {
    ("Swin", 42): None,
    ("Swin", 2025): None,
    ("Swin", 2026): None,
    ("ViT", 42): None,
    ("ViT", 2025): None,
    ("ViT", 2026): None,
}


# ==================================================================================================
# 4. Resolve the development-only Notebook 06 directory
# ==================================================================================================

REVISION_ROOT_CANDIDATES = [
    Path(
        (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026')
    ),
    Path(
        (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026')
    ),
]


REVISION_ROOT = next(
    (
        path
        for path in REVISION_ROOT_CANDIDATES
        if path.exists()
    ),
    None,
)


if REVISION_ROOT is None:
    checked_paths = "\n".join(
        str(path)
        for path in REVISION_ROOT_CANDIDATES
    )

    raise FileNotFoundError(
        "Could not locate the revision_2026 directory.\n"
        "Checked:\n"
        + checked_paths
    )


RA06_CANDIDATES = [
    REVISION_ROOT
    / "06_backbone_benchmark_development_only_v1",

    REVISION_ROOT
    / "06_backbone_benchmark_development_only",
]


RA06 = next(
    (
        path
        for path in RA06_CANDIDATES
        if path.exists()
    ),
    None,
)


if RA06 is None:
    fallback_candidates = sorted(
        [
            path
            for path in REVISION_ROOT.iterdir()
            if (
                path.is_dir()
                and
                path.name.startswith("06_")
            )
        ],
        key=lambda path: (
            path.stat().st_mtime,
            path.name,
        ),
    )

    if fallback_candidates:
        RA06 = fallback_candidates[-1]


if RA06 is None:
    raise FileNotFoundError(
        "Could not locate the Notebook 06 development-only "
        "backbone benchmark directory."
    )


OUTPUT_ROOT = (
    REVISION_ROOT
    / "12_development_oof_probability_calibration_audit_v1"
)

SOURCE_DATA_DIR = (
    OUTPUT_ROOT
    / "source_data"
)

MANIFEST_DIR = (
    OUTPUT_ROOT
    / "manifests"
)

FIGURE_ROOT = (
    OUTPUT_ROOT
    / "figures"
)

PNG_DIR = (
    FIGURE_ROOT
    / "png"
)

SVG_DIR = (
    FIGURE_ROOT
    / "svg"
)

PDF_DIR = (
    FIGURE_ROOT
    / "pdf"
)


for directory in [
    OUTPUT_ROOT,
    SOURCE_DATA_DIR,
    MANIFEST_DIR,
    FIGURE_ROOT,
    PNG_DIR,
    SVG_DIR,
    PDF_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print("=" * 125)
print("12_development_oof_probability_calibration_audit")
print("=" * 125)

print("Revision root:", REVISION_ROOT)
print("Notebook 06 root:", RA06)
print("Output root:", OUTPUT_ROOT)


# ==================================================================================================
# 5. Development-only safety checks
# ==================================================================================================

RA06_NORMALIZED = str(RA06).replace(
    "\\",
    "/",
).lower()


if "development_only" not in RA06_NORMALIZED:
    raise RuntimeError(
        "The selected Notebook 06 directory is not explicitly "
        "identified as development-only:\n"
        + str(RA06)
    )


DISALLOWED_PATH_TERMS = [
    "/test_set/",
    "official_test",
    "official-partition",
    "official_partition",
    "/09_locked",
]


for disallowed_term in DISALLOWED_PATH_TERMS:
    if disallowed_term in RA06_NORMALIZED:
        raise RuntimeError(
            "A disallowed official-partition path was selected:\n"
            + str(RA06)
        )


TRAINING_PERFORMED = False
CHECKPOINTS_LOADED = False
RAW_IMAGES_ACCESSED = False
MODEL_INFERENCE_PERFORMED = False
OFFICIAL_PARTITION_ACCESSED = False
CALIBRATION_MODEL_FITTED = False
TEMPERATURE_SCALING_PERFORMED = False
THRESHOLD_OPTIMIZED = False
MODEL_SELECTION_PERFORMED = False
PRIMARY_ANALYSIS_CHANGED = False


print("")
print("Safety status:")
print("  Training performed:", TRAINING_PERFORMED)
print("  Checkpoints loaded:", CHECKPOINTS_LOADED)
print("  Raw images accessed:", RAW_IMAGES_ACCESSED)
print("  Model inference performed:", MODEL_INFERENCE_PERFORMED)
print("  Official partition accessed:", OFFICIAL_PARTITION_ACCESSED)
print("  Calibration model fitted:", CALIBRATION_MODEL_FITTED)
print("  Temperature scaling performed:", TEMPERATURE_SCALING_PERFORMED)
print("  Threshold optimized:", THRESHOLD_OPTIMIZED)
print("  Model selection performed:", MODEL_SELECTION_PERFORMED)
print("  Primary analysis changed:", PRIMARY_ANALYSIS_CHANGED)


# ==================================================================================================
# 6. Atomic file helpers
# ==================================================================================================

def atomic_csv_save(
    dataframe,
    path,
    index=False,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    dataframe.to_csv(
        temporary_path,
        index=index,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_json_save(
    payload,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

        file.flush()
        os.fsync(
            file.fileno()
        )

    os.replace(
        temporary_path,
        path,
    )


def atomic_text_save(
    text,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(text)

        file.flush()
        os.fsync(
            file.fileno()
        )

    os.replace(
        temporary_path,
        path,
    )


def file_sha256(
    path,
    block_size=8 * 1024 * 1024,
):
    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            block = file.read(
                block_size
            )

            if not block:
                break

            hasher.update(
                block
            )

    return hasher.hexdigest()


def json_fingerprint(
    payload,
):
    canonical = json.dumps(
        payload,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
        ensure_ascii=False,
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        canonical
    ).hexdigest()


for abandoned_file in OUTPUT_ROOT.rglob(
    "*.temporary"
):
    try:
        abandoned_file.unlink()

    except Exception:
        pass


# ==================================================================================================
# 7. Freeze the post hoc calibration-audit plan
# ==================================================================================================

CALIBRATION_METRIC_DEFINITIONS = {
    "nll": (
        "Mean negative natural logarithm of the probability "
        "assigned to the observed class."
    ),
    "multiclass_brier": (
        "Mean sum, across four classes, of squared differences "
        "between predicted probabilities and one-hot outcomes."
    ),
    "ranked_probability_score": (
        "Mean cumulative-probability squared error across the "
        "three ordinal thresholds, normalized by C-1."
    ),
    "top_label_ece": (
        "Fixed-bin expected calibration error using maximum "
        "predicted probability and exact-class correctness."
    ),
    "top_label_mce": (
        "Maximum absolute top-label calibration gap among "
        "nonempty bins."
    ),
    "top_label_rmsce": (
        "Root weighted mean squared top-label calibration gap."
    ),
    "classwise_ece_macro": (
        "Mean of four one-vs-rest classwise expected "
        "calibration errors."
    ),
    "confidence_minus_accuracy": (
        "Mean maximum probability minus exact-class accuracy; "
        "positive values indicate average overconfidence."
    ),
}


ANALYSIS_SPECIFICATION = {
    "protocol_version": (
        "development_oof_probability_calibration_audit_v1"
    ),
    "analysis_role": (
        "post_hoc development-only probability-calibration audit"
    ),
    "source_stage": (
        "Notebook 06 uniform development-only selected-system benchmark"
    ),
    "systems": [
        "Swin",
        "ViT",
    ],
    "training_seeds": TRAINING_SEEDS,
    "evaluation_levels": [
        "image",
        "patient",
    ],
    "patient_reference": (
        "modal_low"
    ),
    "patient_aggregation": (
        "mean_probability_argmax"
    ),
    "primary_ece_bins": (
        PRIMARY_ECE_BINS
    ),
    "ece_bin_sensitivity": (
        ECE_BIN_SENSITIVITY
    ),
    "ece_bins": (
        "equal-width bins over [0,1]; "
        "bin index=floor(probability*n_bins), clipped to n_bins-1"
    ),
    "brier_definition": (
        "mean sum across four classes of squared probability error"
    ),
    "logarithm_for_nll": (
        "natural logarithm"
    ),
    "probability_clipping_epsilon": (
        PROBABILITY_EPSILON
    ),
    "reliability_diagram_source": (
        "mean of the three seed-specific OOF probability vectors"
    ),
    "bootstrap": {
        "replicates": BOOTSTRAP_REPLICATES,
        "seed": BOOTSTRAP_SEED,
        "sampling_unit": (
            "patient"
        ),
        "image_level_resampling": (
            "patient-cluster resampling; all images from a "
            "sampled patient receive the patient's multiplicity"
        ),
        "patient_level_resampling": (
            "patient resampling"
        ),
        "stratification": (
            "modal-low patient reference"
        ),
        "interval": (
            "2.5th and 97.5th percentiles"
        ),
    },
    "metric_definitions": (
        CALIBRATION_METRIC_DEFINITIONS
    ),
    "guardrails": [
        "No model training",
        "No checkpoint loading",
        "No raw-image access",
        "No official-partition access",
        "No calibration-model fitting",
        "No temperature scaling",
        "No isotonic regression",
        "No threshold optimization",
        "No model-system reselection",
        "No aggregation-rule reselection",
        "No patient-reference reselection",
        "No change to the locked official-partition analysis",
    ],
}


ANALYSIS_PLAN_FINGERPRINT = json_fingerprint(
    ANALYSIS_SPECIFICATION
)


ANALYSIS_PLAN_PATH = (
    OUTPUT_ROOT
    / "analysis_plan_development_oof_calibration_audit.json"
)


if ANALYSIS_PLAN_PATH.exists():
    with open(
        ANALYSIS_PLAN_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        existing_plan = json.load(
            file
        )

    existing_fingerprint = existing_plan.get(
        "analysis_plan_fingerprint"
    )

    if (
        existing_fingerprint
        !=
        ANALYSIS_PLAN_FINGERPRINT
    ):
        raise RuntimeError(
            "The current calibration specification differs from "
            "the existing frozen plan.\n\n"
            "Do not overwrite this output directory. Archive it "
            "or change OUTPUT_ROOT to a new version."
        )

    print("")
    print("Existing calibration-audit plan validated:")
    print(ANALYSIS_PLAN_PATH)

else:
    analysis_plan_payload = {
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "analysis_plan_fingerprint": (
            ANALYSIS_PLAN_FINGERPRINT
        ),
        "analysis_specification": (
            ANALYSIS_SPECIFICATION
        ),
    }

    atomic_json_save(
        analysis_plan_payload,
        ANALYSIS_PLAN_PATH,
    )

    print("")
    print("New calibration-audit plan written:")
    print(ANALYSIS_PLAN_PATH)


print("Calibration-audit plan fingerprint:")
print(ANALYSIS_PLAN_FINGERPRINT)


# ==================================================================================================
# 8. General dataframe helpers
# ==================================================================================================

def normalize_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).strip().lower(),
    ).strip("_")


def find_column(
    dataframe,
    aliases,
):
    normalized_to_original = {
        normalize_name(column): column
        for column in dataframe.columns
    }

    for alias in aliases:
        normalized_alias = normalize_name(
            alias
        )

        if normalized_alias in normalized_to_original:
            return normalized_to_original[
                normalized_alias
            ]

    return None


def identify_probability_columns(
    dataframe,
):
    normalized_to_original = {
        normalize_name(column): column
        for column in dataframe.columns
    }

    output_columns = []

    for class_id in CLASS_IDS:
        aliases = [
            f"prob_{class_id}",
            f"probability_{class_id}",
            f"p_{class_id}",
            f"class_{class_id}_prob",
            f"class_{class_id}_probability",
            f"mayo_{class_id}_prob",
            f"mayo_{class_id}_probability",
        ]

        selected_column = None

        for alias in aliases:
            normalized_alias = normalize_name(
                alias
            )

            if normalized_alias in normalized_to_original:
                selected_column = (
                    normalized_to_original[
                        normalized_alias
                    ]
                )
                break

        if selected_column is None:
            suffix_patterns = [
                re.compile(
                    rf".*prob_?{class_id}$"
                ),
                re.compile(
                    rf".*probability_?{class_id}$"
                ),
            ]

            matches = []

            for normalized_column, original_column in (
                normalized_to_original.items()
            ):
                if any(
                    pattern.fullmatch(
                        normalized_column
                    )
                    for pattern in suffix_patterns
                ):
                    matches.append(
                        original_column
                    )

            if len(matches) == 1:
                selected_column = matches[0]

        if selected_column is None:
            return []

        output_columns.append(
            selected_column
        )

    return output_columns


def infer_system_from_text(
    value,
):
    text = str(value).lower()

    if "swin" in text:
        return "Swin"

    vit_patterns = [
        r"(^|[/_\-])vit([/_\-]|$)",
        r"vit_l[0-9]",
        r"vision_transformer",
    ]

    if any(
        re.search(
            pattern,
            text,
        )
        for pattern in vit_patterns
    ):
        return "ViT"

    return None


def infer_seed_from_text(
    value,
):
    text = str(value).lower()

    match = re.search(
        r"seed[_/\-]?(\d+)",
        text,
    )

    if match is None:
        return None

    return int(
        match.group(1)
    )


def normalize_patient_ids(
    series,
):
    output = series.astype(
        str
    ).str.strip()

    output = output.str.replace(
        r"\.0$",
        "",
        regex=True,
    )

    return output


def normalized_path_string(
    value,
):
    return str(value).replace(
        "\\",
        "/",
    ).strip()


def add_image_key(
    dataframe,
):
    dataframe = dataframe.copy()

    base_names = dataframe[
        "path"
    ].astype(str).map(
        lambda value: Path(
            normalized_path_string(
                value
            )
        ).name
    )

    candidate_key = (
        dataframe[
            "patient_id"
        ].astype(str)
        +
        "::"
        +
        base_names
    )

    if not candidate_key.duplicated().any():
        dataframe[
            "image_key"
        ] = candidate_key

        return dataframe

    full_path_key = (
        dataframe[
            "patient_id"
        ].astype(str)
        +
        "::"
        +
        dataframe[
            "path"
        ].astype(str).map(
            normalized_path_string
        )
    )

    if full_path_key.duplicated().any():
        raise RuntimeError(
            "A development source contains duplicate image keys "
            "after both basename and full-path normalization."
        )

    dataframe[
        "image_key"
    ] = full_path_key

    return dataframe


def standardize_prediction_subset(
    dataframe,
):
    patient_column = find_column(
        dataframe,
        aliases=[
            "patient_id",
            "patient",
            "pid",
            "group_id",
        ],
    )

    true_column = find_column(
        dataframe,
        aliases=[
            "y_true",
            "true_label",
            "image_label",
            "label",
            "target",
            "mes_label",
            "mayo_label",
            "mayo_class",
        ],
    )

    path_column = find_column(
        dataframe,
        aliases=[
            "path",
            "image_path",
            "filepath",
            "file_path",
            "filename",
            "image_id",
        ],
    )

    probability_source_columns = (
        identify_probability_columns(
            dataframe
        )
    )

    if patient_column is None:
        return None

    if true_column is None:
        return None

    if (
        len(
            probability_source_columns
        )
        !=
        NUM_CLASSES
    ):
        return None

    output = pd.DataFrame()

    output[
        "patient_id"
    ] = normalize_patient_ids(
        dataframe[
            patient_column
        ]
    )

    output[
        "y_true"
    ] = pd.to_numeric(
        dataframe[
            true_column
        ],
        errors="coerce",
    )

    if path_column is not None:
        output[
            "path"
        ] = dataframe[
            path_column
        ].astype(str)

    else:
        output[
            "path"
        ] = [
            f"row_{row_index:07d}"
            for row_index in range(
                len(dataframe)
            )
        ]

    for class_id, source_column in enumerate(
        probability_source_columns
    ):
        output[
            f"prob_{class_id}"
        ] = pd.to_numeric(
            dataframe[
                source_column
            ],
            errors="coerce",
        )

    output = output.dropna(
        subset=[
            "patient_id",
            "y_true",
            "prob_0",
            "prob_1",
            "prob_2",
            "prob_3",
        ]
    ).copy()

    output[
        "y_true"
    ] = output[
        "y_true"
    ].astype(int)

    invalid_labels = set(
        output[
            "y_true"
        ].unique()
    ) - set(
        CLASS_IDS
    )

    if invalid_labels:
        return None

    probability_matrix = output[
        PROBABILITY_COLUMNS
    ].to_numpy(
        dtype=float
    )

    probability_matrix = np.clip(
        probability_matrix,
        0.0,
        None,
    )

    row_sums = probability_matrix.sum(
        axis=1,
        keepdims=True,
    )

    if np.any(
        row_sums <= 0
    ):
        return None

    probability_matrix = (
        probability_matrix
        / row_sums
    )

    output[
        PROBABILITY_COLUMNS
    ] = probability_matrix

    output[
        "y_pred"
    ] = np.argmax(
        probability_matrix,
        axis=1,
    ).astype(int)

    output = add_image_key(
        output
    )

    return output.reset_index(
        drop=True
    )


# ==================================================================================================
# 9. Modal-low reference and patient aggregation
# ==================================================================================================

def modal_low(
    values,
):
    values = np.asarray(
        values,
        dtype=int,
    )

    counts = np.bincount(
        values,
        minlength=NUM_CLASSES,
    )

    return int(
        np.flatnonzero(
            counts == counts.max()
        )[0]
    )


def derive_patient_probability_table(
    image_dataframe,
):
    rows = []

    for patient_id, group in image_dataframe.groupby(
        "patient_id",
        sort=True,
    ):
        image_labels = group[
            "y_true"
        ].to_numpy(
            dtype=int
        )

        mean_probabilities = group[
            PROBABILITY_COLUMNS
        ].mean(
            axis=0
        ).to_numpy(
            dtype=float
        )

        mean_probabilities = (
            mean_probabilities
            /
            mean_probabilities.sum()
        )

        row = {
            "patient_id": str(
                patient_id
            ),
            "y_true": modal_low(
                image_labels
            ),
            "y_pred": int(
                np.argmax(
                    mean_probabilities
                )
            ),
            "n_images": int(
                len(group)
            ),
        }

        for class_id in CLASS_IDS:
            row[
                f"prob_{class_id}"
            ] = float(
                mean_probabilities[
                    class_id
                ]
            )

        rows.append(
            row
        )

    return pd.DataFrame(
        rows
    ).sort_values(
        "patient_id"
    ).reset_index(
        drop=True
    )


def primary_patient_qwk(
    image_dataframe,
):
    patient_dataframe = (
        derive_patient_probability_table(
            image_dataframe
        )
    )

    qwk = cohen_kappa_score(
        patient_dataframe[
            "y_true"
        ],
        patient_dataframe[
            "y_pred"
        ],
        labels=CLASS_IDS,
        weights="quadratic",
    )

    return (
        float(qwk),
        patient_dataframe,
    )


# ==================================================================================================
# 10. Calibration metric implementation
# ==================================================================================================

def equal_width_bin_ids(
    probabilities,
    n_bins,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    bin_ids = np.floor(
        probabilities
        *
        n_bins
    ).astype(int)

    return np.clip(
        bin_ids,
        0,
        n_bins - 1,
    )


def weighted_mean(
    values,
    weights,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    total_weight = float(
        weights.sum()
    )

    if total_weight <= 0:
        return np.nan

    return float(
        np.sum(
            values
            *
            weights
        )
        /
        total_weight
    )


def build_calibration_precomputation(
    y_true,
    probabilities,
    n_bins,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    probabilities = np.clip(
        probabilities,
        PROBABILITY_EPSILON,
        1.0,
    )

    probabilities = (
        probabilities
        /
        probabilities.sum(
            axis=1,
            keepdims=True,
        )
    )

    one_hot = np.eye(
        NUM_CLASSES,
        dtype=float,
    )[
        y_true
    ]

    predicted_class = np.argmax(
        probabilities,
        axis=1,
    )

    confidence = np.max(
        probabilities,
        axis=1,
    )

    correct = (
        predicted_class
        ==
        y_true
    ).astype(float)

    nll_per_observation = -np.log(
        probabilities[
            np.arange(
                len(y_true)
            ),
            y_true,
        ]
    )

    brier_per_observation = np.sum(
        (
            probabilities
            -
            one_hot
        )
        ** 2,
        axis=1,
    )

    cumulative_probabilities = np.cumsum(
        probabilities[
            :,
            :-1,
        ],
        axis=1,
    )

    cumulative_outcomes = np.column_stack(
        [
            (
                y_true
                <=
                threshold
            ).astype(float)
            for threshold in range(
                NUM_CLASSES - 1
            )
        ]
    )

    ranked_probability_per_observation = (
        np.sum(
            (
                cumulative_probabilities
                -
                cumulative_outcomes
            )
            ** 2,
            axis=1,
        )
        /
        (
            NUM_CLASSES - 1
        )
    )

    top_bin_ids = equal_width_bin_ids(
        confidence,
        n_bins,
    )

    class_bin_ids = np.column_stack(
        [
            equal_width_bin_ids(
                probabilities[
                    :,
                    class_id,
                ],
                n_bins,
            )
            for class_id in CLASS_IDS
        ]
    )

    class_targets = np.column_stack(
        [
            (
                y_true
                ==
                class_id
            ).astype(float)
            for class_id in CLASS_IDS
        ]
    )

    return {
        "y_true": y_true,
        "probabilities": probabilities,
        "predicted_class": predicted_class,
        "confidence": confidence,
        "correct": correct,
        "nll_per_observation": (
            nll_per_observation
        ),
        "brier_per_observation": (
            brier_per_observation
        ),
        "rps_per_observation": (
            ranked_probability_per_observation
        ),
        "top_bin_ids": top_bin_ids,
        "class_bin_ids": class_bin_ids,
        "class_targets": class_targets,
        "n_bins": int(
            n_bins
        ),
    }


def binned_calibration_statistics(
    bin_ids,
    predicted_probability,
    observed_outcome,
    weights,
    n_bins,
):
    bin_ids = np.asarray(
        bin_ids,
        dtype=int,
    )

    predicted_probability = np.asarray(
        predicted_probability,
        dtype=float,
    )

    observed_outcome = np.asarray(
        observed_outcome,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    bin_weight = np.bincount(
        bin_ids,
        weights=weights,
        minlength=n_bins,
    ).astype(float)

    probability_sum = np.bincount(
        bin_ids,
        weights=(
            weights
            *
            predicted_probability
        ),
        minlength=n_bins,
    ).astype(float)

    outcome_sum = np.bincount(
        bin_ids,
        weights=(
            weights
            *
            observed_outcome
        ),
        minlength=n_bins,
    ).astype(float)

    mean_probability = np.divide(
        probability_sum,
        bin_weight,
        out=np.full(
            n_bins,
            np.nan,
            dtype=float,
        ),
        where=bin_weight > 0,
    )

    observed_frequency = np.divide(
        outcome_sum,
        bin_weight,
        out=np.full(
            n_bins,
            np.nan,
            dtype=float,
        ),
        where=bin_weight > 0,
    )

    absolute_gap = np.abs(
        observed_frequency
        -
        mean_probability
    )

    valid_bins = (
        bin_weight > 0
    )

    total_weight = float(
        bin_weight.sum()
    )

    if total_weight <= 0:
        return {
            "bin_weight": bin_weight,
            "mean_probability": mean_probability,
            "observed_frequency": observed_frequency,
            "absolute_gap": absolute_gap,
            "ece": np.nan,
            "mce": np.nan,
            "rmsce": np.nan,
        }

    normalized_bin_weight = (
        bin_weight
        /
        total_weight
    )

    ece = float(
        np.nansum(
            normalized_bin_weight
            *
            absolute_gap
        )
    )

    mce = float(
        np.nanmax(
            absolute_gap[
                valid_bins
            ]
        )
    )

    rmsce = float(
        np.sqrt(
            np.nansum(
                normalized_bin_weight
                *
                (
                    absolute_gap
                    ** 2
                )
            )
        )
    )

    return {
        "bin_weight": bin_weight,
        "mean_probability": mean_probability,
        "observed_frequency": observed_frequency,
        "absolute_gap": absolute_gap,
        "ece": ece,
        "mce": mce,
        "rmsce": rmsce,
    }


def calibration_metrics_from_precomputation(
    precomputed,
    sample_weights=None,
):
    n_observations = len(
        precomputed[
            "y_true"
        ]
    )

    if sample_weights is None:
        sample_weights = np.ones(
            n_observations,
            dtype=float,
        )

    else:
        sample_weights = np.asarray(
            sample_weights,
            dtype=float,
        )

    total_weight = float(
        sample_weights.sum()
    )

    if total_weight <= 0:
        raise ValueError(
            "Calibration sample weights sum to zero."
        )

    nll = weighted_mean(
        precomputed[
            "nll_per_observation"
        ],
        sample_weights,
    )

    brier = weighted_mean(
        precomputed[
            "brier_per_observation"
        ],
        sample_weights,
    )

    ranked_probability_score = weighted_mean(
        precomputed[
            "rps_per_observation"
        ],
        sample_weights,
    )

    accuracy = weighted_mean(
        precomputed[
            "correct"
        ],
        sample_weights,
    )

    mean_confidence = weighted_mean(
        precomputed[
            "confidence"
        ],
        sample_weights,
    )

    top_statistics = binned_calibration_statistics(
        bin_ids=precomputed[
            "top_bin_ids"
        ],
        predicted_probability=precomputed[
            "confidence"
        ],
        observed_outcome=precomputed[
            "correct"
        ],
        weights=sample_weights,
        n_bins=precomputed[
            "n_bins"
        ],
    )

    classwise_ece_values = []

    for class_id in CLASS_IDS:
        class_statistics = (
            binned_calibration_statistics(
                bin_ids=precomputed[
                    "class_bin_ids"
                ][
                    :,
                    class_id,
                ],
                predicted_probability=precomputed[
                    "probabilities"
                ][
                    :,
                    class_id,
                ],
                observed_outcome=precomputed[
                    "class_targets"
                ][
                    :,
                    class_id,
                ],
                weights=sample_weights,
                n_bins=precomputed[
                    "n_bins"
                ],
            )
        )

        classwise_ece_values.append(
            class_statistics[
                "ece"
            ]
        )

    output = {
        "nll": float(
            nll
        ),
        "multiclass_brier": float(
            brier
        ),
        "ranked_probability_score": float(
            ranked_probability_score
        ),
        "top_label_ece": float(
            top_statistics[
                "ece"
            ]
        ),
        "top_label_mce": float(
            top_statistics[
                "mce"
            ]
        ),
        "top_label_rmsce": float(
            top_statistics[
                "rmsce"
            ]
        ),
        "classwise_ece_macro": float(
            np.mean(
                classwise_ece_values
            )
        ),
        "accuracy": float(
            accuracy
        ),
        "mean_confidence": float(
            mean_confidence
        ),
        "confidence_minus_accuracy": float(
            mean_confidence
            -
            accuracy
        ),
    }

    for class_id in CLASS_IDS:
        output[
            f"classwise_ece_{class_id}"
        ] = float(
            classwise_ece_values[
                class_id
            ]
        )

    return output


def reliability_bin_rows(
    precomputed,
    system,
    level,
    scope,
    training_seed,
):
    rows = []

    weights = np.ones(
        len(
            precomputed[
                "y_true"
            ]
        ),
        dtype=float,
    )

    top_statistics = binned_calibration_statistics(
        bin_ids=precomputed[
            "top_bin_ids"
        ],
        predicted_probability=precomputed[
            "confidence"
        ],
        observed_outcome=precomputed[
            "correct"
        ],
        weights=weights,
        n_bins=precomputed[
            "n_bins"
        ],
    )

    n_bins = precomputed[
        "n_bins"
    ]

    for bin_id in range(
        n_bins
    ):
        rows.append(
            {
                "system": system,
                "level": level,
                "scope": scope,
                "training_seed": (
                    training_seed
                ),
                "calibration_type": (
                    "top_label"
                ),
                "class_id": np.nan,
                "class_name": "",
                "n_bins": n_bins,
                "bin_id": bin_id,
                "bin_lower": (
                    bin_id
                    /
                    n_bins
                ),
                "bin_upper": (
                    (
                        bin_id
                        +
                        1
                    )
                    /
                    n_bins
                ),
                "n_observations": int(
                    top_statistics[
                        "bin_weight"
                    ][
                        bin_id
                    ]
                ),
                "mean_predicted_probability": (
                    top_statistics[
                        "mean_probability"
                    ][
                        bin_id
                    ]
                ),
                "observed_frequency": (
                    top_statistics[
                        "observed_frequency"
                    ][
                        bin_id
                    ]
                ),
                "absolute_gap": (
                    top_statistics[
                        "absolute_gap"
                    ][
                        bin_id
                    ]
                ),
            }
        )

    for class_id in CLASS_IDS:
        class_statistics = (
            binned_calibration_statistics(
                bin_ids=precomputed[
                    "class_bin_ids"
                ][
                    :,
                    class_id,
                ],
                predicted_probability=precomputed[
                    "probabilities"
                ][
                    :,
                    class_id,
                ],
                observed_outcome=precomputed[
                    "class_targets"
                ][
                    :,
                    class_id,
                ],
                weights=weights,
                n_bins=n_bins,
            )
        )

        for bin_id in range(
            n_bins
        ):
            rows.append(
                {
                    "system": system,
                    "level": level,
                    "scope": scope,
                    "training_seed": (
                        training_seed
                    ),
                    "calibration_type": (
                        "classwise_one_vs_rest"
                    ),
                    "class_id": class_id,
                    "class_name": CLASS_NAMES[
                        class_id
                    ],
                    "n_bins": n_bins,
                    "bin_id": bin_id,
                    "bin_lower": (
                        bin_id
                        /
                        n_bins
                    ),
                    "bin_upper": (
                        (
                            bin_id
                            +
                            1
                        )
                        /
                        n_bins
                    ),
                    "n_observations": int(
                        class_statistics[
                            "bin_weight"
                        ][
                            bin_id
                        ]
                    ),
                    "mean_predicted_probability": (
                        class_statistics[
                            "mean_probability"
                        ][
                            bin_id
                        ]
                    ),
                    "observed_frequency": (
                        class_statistics[
                            "observed_frequency"
                        ][
                            bin_id
                        ]
                    ),
                    "absolute_gap": (
                        class_statistics[
                            "absolute_gap"
                        ][
                            bin_id
                        ]
                    ),
                }
            )

    return rows


# ==================================================================================================
# 11. Discover the six exact development OOF probability sources
# ==================================================================================================

def source_system_column(
    dataframe,
):
    return find_column(
        dataframe,
        aliases=[
            "system",
            "system_name",
            "backbone",
            "backbone_name",
            "model",
            "model_name",
            "candidate",
            "candidate_name",
            "representative_configuration",
            "configuration",
        ],
    )


def source_seed_column(
    dataframe,
):
    return find_column(
        dataframe,
        aliases=[
            "training_seed",
            "seed",
            "random_seed",
        ],
    )


def load_source_subset(
    path,
    system,
    training_seed,
):
    raw_dataframe = pd.read_csv(
        path,
        low_memory=False,
    )

    system_column = source_system_column(
        raw_dataframe
    )

    seed_column = source_seed_column(
        raw_dataframe
    )

    subset = raw_dataframe.copy()

    if system_column is not None:
        parsed_systems = subset[
            system_column
        ].map(
            infer_system_from_text
        )

        if (
            parsed_systems
            ==
            system
        ).any():
            subset = subset.loc[
                parsed_systems
                ==
                system
            ].copy()

    if seed_column is not None:
        numeric_seed = pd.to_numeric(
            subset[
                seed_column
            ],
            errors="coerce",
        )

        if (
            numeric_seed
            ==
            training_seed
        ).any():
            subset = subset.loc[
                numeric_seed
                ==
                training_seed
            ].copy()

    standardized = (
        standardize_prediction_subset(
            subset
        )
    )

    return standardized


def discover_development_sources():
    discovery_rows = []
    accepted_records = []

    for csv_path in sorted(
        RA06.rglob(
            "*.csv"
        )
    ):
        normalized_path = str(
            csv_path
        ).replace(
            "\\",
            "/",
        ).lower()

        if any(
            term in normalized_path
            for term in DISALLOWED_PATH_TERMS
        ):
            continue

        file_name_lower = (
            csv_path.name.lower()
        )

        exclusion_terms = [
            "bootstrap",
            "confusion",
            "history",
            "manifest",
            "decision",
        ]

        if (
            any(
                term in file_name_lower
                for term in exclusion_terms
            )
            and
            "prediction"
            not in file_name_lower
        ):
            continue

        try:
            header_dataframe = pd.read_csv(
                csv_path,
                nrows=8,
            )

        except Exception:
            continue

        if (
            len(
                identify_probability_columns(
                    header_dataframe
                )
            )
            !=
            NUM_CLASSES
        ):
            continue

        inferred_system = infer_system_from_text(
            normalized_path
        )

        inferred_seed = infer_seed_from_text(
            normalized_path
        )

        try:
            full_dataframe = pd.read_csv(
                csv_path,
                low_memory=False,
            )

        except Exception as read_error:
            discovery_rows.append(
                {
                    "path": str(
                        csv_path
                    ),
                    "status": "read_error",
                    "error": repr(
                        read_error
                    ),
                }
            )

            continue

        system_column = source_system_column(
            full_dataframe
        )

        seed_column = source_seed_column(
            full_dataframe
        )

        if system_column is not None:
            candidate_systems = [
                value
                for value in (
                    full_dataframe[
                        system_column
                    ]
                    .map(
                        infer_system_from_text
                    )
                    .dropna()
                    .unique()
                    .tolist()
                )
                if value in {
                    "Swin",
                    "ViT",
                }
            ]

        elif inferred_system is not None:
            candidate_systems = [
                inferred_system
            ]

        else:
            candidate_systems = []

        if seed_column is not None:
            candidate_seeds = (
                pd.to_numeric(
                    full_dataframe[
                        seed_column
                    ],
                    errors="coerce",
                )
                .dropna()
                .astype(int)
                .unique()
                .tolist()
            )

        elif inferred_seed is not None:
            candidate_seeds = [
                inferred_seed
            ]

        else:
            candidate_seeds = []

        for system in candidate_systems:
            for training_seed in candidate_seeds:
                if training_seed not in TRAINING_SEEDS:
                    continue

                subset = full_dataframe.copy()

                if system_column is not None:
                    parsed_systems = subset[
                        system_column
                    ].map(
                        infer_system_from_text
                    )

                    subset = subset.loc[
                        parsed_systems
                        ==
                        system
                    ].copy()

                if seed_column is not None:
                    numeric_seed = pd.to_numeric(
                        subset[
                            seed_column
                        ],
                        errors="coerce",
                    )

                    subset = subset.loc[
                        numeric_seed
                        ==
                        training_seed
                    ].copy()

                standardized = (
                    standardize_prediction_subset(
                        subset
                    )
                )

                if standardized is None:
                    continue

                n_images = int(
                    len(
                        standardized
                    )
                )

                n_patients = int(
                    standardized[
                        "patient_id"
                    ].nunique()
                )

                if (
                    n_images
                    !=
                    EXPECTED_DEVELOPMENT_IMAGES
                    or
                    n_patients
                    !=
                    EXPECTED_DEVELOPMENT_PATIENTS
                ):
                    continue

                try:
                    observed_qwk, _ = (
                        primary_patient_qwk(
                            standardized
                        )
                    )

                except Exception:
                    continue

                expected_qwk = (
                    EXPECTED_PATIENT_QWK[
                        (
                            system,
                            training_seed,
                        )
                    ]
                )

                qwk_difference = abs(
                    observed_qwk
                    -
                    expected_qwk
                )

                score = (
                    -10000.0
                    *
                    qwk_difference
                )

                if "oof" in file_name_lower:
                    score += 40

                if "image" in file_name_lower:
                    score += 20

                if "prediction" in file_name_lower:
                    score += 20

                if system.lower() in normalized_path:
                    score += 10

                if str(
                    training_seed
                ) in normalized_path:
                    score += 10

                status = (
                    "accepted"
                    if
                    qwk_difference
                    <=
                    SOURCE_QWK_TOLERANCE
                    else
                    "qwk_mismatch"
                )

                discovery_rows.append(
                    {
                        "path": str(
                            csv_path
                        ),
                        "system": system,
                        "training_seed": int(
                            training_seed
                        ),
                        "n_images": n_images,
                        "n_patients": n_patients,
                        "observed_patient_qwk": (
                            observed_qwk
                        ),
                        "expected_patient_qwk": (
                            expected_qwk
                        ),
                        "absolute_qwk_difference": (
                            qwk_difference
                        ),
                        "score": score,
                        "status": status,
                        "error": "",
                    }
                )

                if status == "accepted":
                    accepted_records.append(
                        {
                            "path": csv_path,
                            "system": system,
                            "training_seed": int(
                                training_seed
                            ),
                            "score": score,
                            "qwk_difference": (
                                qwk_difference
                            ),
                            "dataframe": standardized,
                        }
                    )

    return (
        pd.DataFrame(
            discovery_rows
        ),
        accepted_records,
    )


source_discovery_df, accepted_sources = (
    discover_development_sources()
)


atomic_csv_save(
    source_discovery_df,
    MANIFEST_DIR
    / "development_oof_source_discovery.csv",
    index=False,
)


selected_sources = {}


for system in [
    "Swin",
    "ViT",
]:
    for training_seed in TRAINING_SEEDS:
        override_path = (
            DEVELOPMENT_FILE_OVERRIDES[
                (
                    system,
                    training_seed,
                )
            ]
        )

        if override_path is not None:
            override_path = Path(
                override_path
            )

            normalized_override = str(
                override_path
            ).replace(
                "\\",
                "/",
            ).lower()

            if any(
                term in normalized_override
                for term in DISALLOWED_PATH_TERMS
            ):
                raise RuntimeError(
                    "The override path contains a disallowed "
                    "official-partition term:\n"
                    + str(
                        override_path
                    )
                )

            standardized = load_source_subset(
                override_path,
                system=system,
                training_seed=training_seed,
            )

            if standardized is None:
                raise RuntimeError(
                    "Could not standardize override source:\n"
                    + str(
                        override_path
                    )
                )

            observed_qwk, _ = (
                primary_patient_qwk(
                    standardized
                )
            )

            expected_qwk = (
                EXPECTED_PATIENT_QWK[
                    (
                        system,
                        training_seed,
                    )
                ]
            )

            if abs(
                observed_qwk
                -
                expected_qwk
            ) > SOURCE_QWK_TOLERANCE:
                raise RuntimeError(
                    "Override source did not reproduce the "
                    "expected Notebook 06 patient QWK.\n"
                    f"System: {system}\n"
                    f"Seed: {training_seed}\n"
                    f"Observed: {observed_qwk:.6f}\n"
                    f"Expected: {expected_qwk:.6f}"
                )

            selected_sources[
                (
                    system,
                    training_seed,
                )
            ] = {
                "path": override_path,
                "dataframe": standardized,
            }

            continue

        matching_sources = [
            record
            for record in accepted_sources
            if (
                record[
                    "system"
                ]
                ==
                system
                and
                record[
                    "training_seed"
                ]
                ==
                training_seed
            )
        ]

        matching_sources = sorted(
            matching_sources,
            key=lambda record: (
                record[
                    "qwk_difference"
                ],
                -record[
                    "score"
                ],
                str(
                    record[
                        "path"
                    ]
                ),
            ),
        )

        if not matching_sources:
            print("")
            print(
                "Automatic source discovery failed for:",
                system,
                training_seed,
            )

            if not source_discovery_df.empty:
                display(
                    source_discovery_df.sort_values(
                        [
                            "system",
                            "training_seed",
                            "score",
                        ],
                        ascending=[
                            True,
                            True,
                            False,
                        ],
                    ).head(
                        50
                    )
                )

            raise FileNotFoundError(
                "Could not identify a complete development OOF "
                "probability source for "
                f"{system}, seed {training_seed}.\n\n"
                "Inspect:\n"
                + str(
                    MANIFEST_DIR
                    / "development_oof_source_discovery.csv"
                )
                + "\n\n"
                "Then set the corresponding "
                "DEVELOPMENT_FILE_OVERRIDES value."
            )

        selected_record = (
            matching_sources[0]
        )

        selected_sources[
            (
                system,
                training_seed,
            )
        ] = {
            "path": selected_record[
                "path"
            ],
            "dataframe": selected_record[
                "dataframe"
            ],
        }


# ==================================================================================================
# 12. Validate selected source counts, support, and QWK
# ==================================================================================================

source_manifest_rows = []


for (
    system,
    training_seed,
), source_record in selected_sources.items():
    source_path = Path(
        source_record[
            "path"
        ]
    )

    source_dataframe = source_record[
        "dataframe"
    ]

    observed_qwk, patient_dataframe = (
        primary_patient_qwk(
            source_dataframe
        )
    )

    observed_support = (
        patient_dataframe[
            "y_true"
        ]
        .value_counts()
        .sort_index()
        .reindex(
            CLASS_IDS,
            fill_value=0,
        )
        .to_dict()
    )

    if observed_support != EXPECTED_MODAL_LOW_SUPPORT:
        raise RuntimeError(
            "Modal-low support differs from the frozen "
            "development reference.\n"
            f"System: {system}\n"
            f"Seed: {training_seed}\n"
            f"Expected: {EXPECTED_MODAL_LOW_SUPPORT}\n"
            f"Observed: {observed_support}"
        )

    source_manifest_rows.append(
        {
            "system": system,
            "training_seed": training_seed,
            "path": str(
                source_path
            ),
            "n_images": int(
                len(
                    source_dataframe
                )
            ),
            "n_patients": int(
                source_dataframe[
                    "patient_id"
                ].nunique()
            ),
            "patient_qwk": float(
                observed_qwk
            ),
            "size_bytes": int(
                source_path.stat().st_size
            ),
            "sha256": file_sha256(
                source_path
            ),
        }
    )


source_manifest_df = pd.DataFrame(
    source_manifest_rows
).sort_values(
    [
        "system",
        "training_seed",
    ]
).reset_index(
    drop=True
)


atomic_csv_save(
    source_manifest_df,
    MANIFEST_DIR
    / "calibration_source_manifest.csv",
    index=False,
)


print("")
print("=" * 125)
print("SELECTED DEVELOPMENT OOF SOURCES")
print("=" * 125)

display(
    source_manifest_df
)


# ==================================================================================================
# 13. Verify image/reference alignment across all six sources
# ==================================================================================================

canonical_source = selected_sources[
    (
        "Swin",
        42,
    )
][
    "dataframe"
][
    [
        "image_key",
        "patient_id",
        "path",
        "y_true",
    ]
].copy()


canonical_source = canonical_source.sort_values(
    "image_key"
).reset_index(
    drop=True
)


for source_key, source_record in selected_sources.items():
    source_dataframe = source_record[
        "dataframe"
    ][
        [
            "image_key",
            "patient_id",
            "y_true",
        ]
    ].copy()

    comparison = canonical_source.merge(
        source_dataframe,
        on="image_key",
        how="inner",
        suffixes=(
            "_canonical",
            "_source",
        ),
        validate="one_to_one",
    )

    if (
        len(
            comparison
        )
        !=
        EXPECTED_DEVELOPMENT_IMAGES
    ):
        raise RuntimeError(
            "OOF image sets differ across saved sources.\n"
            f"Source: {source_key}\n"
            f"Matched images: {len(comparison)}"
        )

    patient_disagreements = int(
        (
            comparison[
                "patient_id_canonical"
            ]
            !=
            comparison[
                "patient_id_source"
            ]
        ).sum()
    )

    label_disagreements = int(
        (
            comparison[
                "y_true_canonical"
            ]
            !=
            comparison[
                "y_true_source"
            ]
        ).sum()
    )

    if (
        patient_disagreements > 0
        or
        label_disagreements > 0
    ):
        raise RuntimeError(
            "OOF image metadata differ across sources.\n"
            f"Source: {source_key}\n"
            f"Patient disagreements: {patient_disagreements}\n"
            f"Label disagreements: {label_disagreements}"
        )


# ==================================================================================================
# 14. Build the three-seed OOF probability ensemble for each system
# ==================================================================================================

def build_seed_ensemble(
    system,
):
    merged = None

    for training_seed in TRAINING_SEEDS:
        source_dataframe = selected_sources[
            (
                system,
                training_seed,
            )
        ][
            "dataframe"
        ].copy()

        seed_dataframe = source_dataframe[
            [
                "image_key",
                "patient_id",
                "path",
                "y_true",
            ]
            +
            PROBABILITY_COLUMNS
        ].copy()

        rename_map = {
            probability_column: (
                probability_column
                +
                "_seed_"
                +
                str(
                    training_seed
                )
            )
            for probability_column in PROBABILITY_COLUMNS
        }

        seed_dataframe = (
            seed_dataframe.rename(
                columns=rename_map
            )
        )

        if merged is None:
            merged = seed_dataframe

        else:
            merged = merged.merge(
                seed_dataframe,
                on=[
                    "image_key",
                    "patient_id",
                    "path",
                    "y_true",
                ],
                how="inner",
                validate="one_to_one",
            )

    if (
        len(
            merged
        )
        !=
        EXPECTED_DEVELOPMENT_IMAGES
    ):
        raise RuntimeError(
            "The three seed-specific OOF files could not be "
            "aligned into one 9,590-image ensemble for "
            + system
        )

    probability_arrays = []

    for training_seed in TRAINING_SEEDS:
        seed_probability_columns = [
            probability_column
            +
            "_seed_"
            +
            str(
                training_seed
            )
            for probability_column in PROBABILITY_COLUMNS
        ]

        probability_arrays.append(
            merged[
                seed_probability_columns
            ].to_numpy(
                dtype=float
            )
        )

    ensemble_probabilities = np.mean(
        np.stack(
            probability_arrays,
            axis=0,
        ),
        axis=0,
    )

    ensemble_probabilities = (
        ensemble_probabilities
        /
        ensemble_probabilities.sum(
            axis=1,
            keepdims=True,
        )
    )

    output = merged[
        [
            "image_key",
            "patient_id",
            "path",
            "y_true",
        ]
    ].copy()

    output[
        PROBABILITY_COLUMNS
    ] = ensemble_probabilities

    output[
        "y_pred"
    ] = np.argmax(
        ensemble_probabilities,
        axis=1,
    ).astype(int)

    return output.sort_values(
        "image_key"
    ).reset_index(
        drop=True
    )


seed_ensemble_image_dfs = {
    system: build_seed_ensemble(
        system
    )
    for system in [
        "Swin",
        "ViT",
    ]
}


seed_ensemble_patient_dfs = {
    system: derive_patient_probability_table(
        seed_ensemble_image_dfs[
            system
        ]
    )
    for system in [
        "Swin",
        "ViT",
    ]
}


for system in [
    "Swin",
    "ViT",
]:
    atomic_csv_save(
        seed_ensemble_image_dfs[
            system
        ],
        SOURCE_DATA_DIR
        /
        (
            system.lower()
            +
            "_development_oof_three_seed_probability_ensemble_images.csv"
        ),
        index=False,
    )

    atomic_csv_save(
        seed_ensemble_patient_dfs[
            system
        ],
        SOURCE_DATA_DIR
        /
        (
            system.lower()
            +
            "_development_oof_three_seed_probability_ensemble_patients.csv"
        ),
        index=False,
    )


# ==================================================================================================
# 15. Calculate calibration metrics for every system and seed
# ==================================================================================================

PER_SEED_METRIC_ROWS = []
PER_SEED_RELIABILITY_ROWS = []
PER_SEED_CLASSWISE_ECE_ROWS = []


for system in [
    "Swin",
    "ViT",
]:
    for training_seed in TRAINING_SEEDS:
        image_dataframe = selected_sources[
            (
                system,
                training_seed,
            )
        ][
            "dataframe"
        ]

        patient_dataframe = (
            derive_patient_probability_table(
                image_dataframe
            )
        )

        level_data = {
            "image": {
                "y_true": image_dataframe[
                    "y_true"
                ].to_numpy(
                    dtype=int
                ),
                "probabilities": image_dataframe[
                    PROBABILITY_COLUMNS
                ].to_numpy(
                    dtype=float
                ),
                "n_observations": int(
                    len(
                        image_dataframe
                    )
                ),
            },
            "patient": {
                "y_true": patient_dataframe[
                    "y_true"
                ].to_numpy(
                    dtype=int
                ),
                "probabilities": patient_dataframe[
                    PROBABILITY_COLUMNS
                ].to_numpy(
                    dtype=float
                ),
                "n_observations": int(
                    len(
                        patient_dataframe
                    )
                ),
            },
        }

        for level, level_record in level_data.items():
            precomputed = (
                build_calibration_precomputation(
                    y_true=level_record[
                        "y_true"
                    ],
                    probabilities=level_record[
                        "probabilities"
                    ],
                    n_bins=PRIMARY_ECE_BINS,
                )
            )

            metrics = (
                calibration_metrics_from_precomputation(
                    precomputed
                )
            )

            metric_row = {
                "system": system,
                "training_seed": (
                    training_seed
                ),
                "level": level,
                "n_observations": (
                    level_record[
                        "n_observations"
                    ]
                ),
                "ece_bins": (
                    PRIMARY_ECE_BINS
                ),
            }

            metric_row.update(
                metrics
            )

            PER_SEED_METRIC_ROWS.append(
                metric_row
            )

            for class_id in CLASS_IDS:
                PER_SEED_CLASSWISE_ECE_ROWS.append(
                    {
                        "system": system,
                        "training_seed": (
                            training_seed
                        ),
                        "level": level,
                        "class_id": class_id,
                        "class_name": (
                            CLASS_NAMES[
                                class_id
                            ]
                        ),
                        "classwise_ece": (
                            metrics[
                                f"classwise_ece_{class_id}"
                            ]
                        ),
                        "ece_bins": (
                            PRIMARY_ECE_BINS
                        ),
                    }
                )

            PER_SEED_RELIABILITY_ROWS.extend(
                reliability_bin_rows(
                    precomputed=precomputed,
                    system=system,
                    level=level,
                    scope="individual_seed",
                    training_seed=training_seed,
                )
            )


calibration_by_seed_df = pd.DataFrame(
    PER_SEED_METRIC_ROWS
)


classwise_ece_by_seed_df = pd.DataFrame(
    PER_SEED_CLASSWISE_ECE_ROWS
)


reliability_bins_by_seed_df = pd.DataFrame(
    PER_SEED_RELIABILITY_ROWS
)


atomic_csv_save(
    calibration_by_seed_df,
    OUTPUT_ROOT
    / "development_calibration_metrics_by_seed.csv",
    index=False,
)


atomic_csv_save(
    classwise_ece_by_seed_df,
    OUTPUT_ROOT
    / "development_classwise_ece_by_seed.csv",
    index=False,
)


atomic_csv_save(
    reliability_bins_by_seed_df,
    SOURCE_DATA_DIR
    / "development_reliability_bins_by_seed.csv",
    index=False,
)


# ==================================================================================================
# 16. Summarize mean ± SD across seeds
# ==================================================================================================

SUMMARY_METRIC_COLUMNS = [
    "nll",
    "multiclass_brier",
    "ranked_probability_score",
    "top_label_ece",
    "top_label_mce",
    "top_label_rmsce",
    "classwise_ece_macro",
    "accuracy",
    "mean_confidence",
    "confidence_minus_accuracy",
]


summary_rows = []


for (
    system,
    level,
), group_dataframe in calibration_by_seed_df.groupby(
    [
        "system",
        "level",
    ],
    sort=False,
):
    row = {
        "system": system,
        "level": level,
        "n_seeds": int(
            group_dataframe[
                "training_seed"
            ].nunique()
        ),
        "n_observations_per_seed": int(
            group_dataframe[
                "n_observations"
            ].iloc[
                0
            ]
        ),
        "ece_bins": (
            PRIMARY_ECE_BINS
        ),
    }

    for metric_name in SUMMARY_METRIC_COLUMNS:
        values = group_dataframe[
            metric_name
        ].to_numpy(
            dtype=float
        )

        row[
            metric_name
            +
            "_mean"
        ] = float(
            np.mean(
                values
            )
        )

        row[
            metric_name
            +
            "_std"
        ] = float(
            np.std(
                values,
                ddof=1,
            )
        )

        row[
            metric_name
            +
            "_min"
        ] = float(
            np.min(
                values
            )
        )

        row[
            metric_name
            +
            "_max"
        ] = float(
            np.max(
                values
            )
        )

    summary_rows.append(
        row
    )


calibration_summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    [
        "level",
        "system",
    ]
).reset_index(
    drop=True
)


atomic_csv_save(
    calibration_summary_df,
    OUTPUT_ROOT
    / "development_calibration_summary_across_seeds.csv",
    index=False,
)


classwise_summary_rows = []


for (
    system,
    level,
    class_id,
    class_name,
), group_dataframe in (
    classwise_ece_by_seed_df.groupby(
        [
            "system",
            "level",
            "class_id",
            "class_name",
        ],
        sort=False,
    )
):
    values = group_dataframe[
        "classwise_ece"
    ].to_numpy(
        dtype=float
    )

    classwise_summary_rows.append(
        {
            "system": system,
            "level": level,
            "class_id": class_id,
            "class_name": class_name,
            "n_seeds": int(
                len(
                    values
                )
            ),
            "classwise_ece_mean": float(
                np.mean(
                    values
                )
            ),
            "classwise_ece_std": float(
                np.std(
                    values,
                    ddof=1,
                )
            ),
            "classwise_ece_min": float(
                np.min(
                    values
                )
            ),
            "classwise_ece_max": float(
                np.max(
                    values
                )
            ),
        }
    )


classwise_ece_summary_df = pd.DataFrame(
    classwise_summary_rows
).sort_values(
    [
        "level",
        "system",
        "class_id",
    ]
).reset_index(
    drop=True
)


atomic_csv_save(
    classwise_ece_summary_df,
    OUTPUT_ROOT
    / "development_classwise_ece_summary_across_seeds.csv",
    index=False,
)


# ==================================================================================================
# 17. Seed-ensemble calibration metrics and reliability bins
# ==================================================================================================

SEED_ENSEMBLE_METRIC_ROWS = []
SEED_ENSEMBLE_RELIABILITY_ROWS = []
SEED_ENSEMBLE_CLASSWISE_ECE_ROWS = []

seed_ensemble_precomputations = {}


for system in [
    "Swin",
    "ViT",
]:
    image_dataframe = (
        seed_ensemble_image_dfs[
            system
        ]
    )

    patient_dataframe = (
        seed_ensemble_patient_dfs[
            system
        ]
    )

    level_data = {
        "image": {
            "y_true": image_dataframe[
                "y_true"
            ].to_numpy(
                dtype=int
            ),
            "probabilities": image_dataframe[
                PROBABILITY_COLUMNS
            ].to_numpy(
                dtype=float
            ),
            "n_observations": int(
                len(
                    image_dataframe
                )
            ),
        },
        "patient": {
            "y_true": patient_dataframe[
                "y_true"
            ].to_numpy(
                dtype=int
            ),
            "probabilities": patient_dataframe[
                PROBABILITY_COLUMNS
            ].to_numpy(
                dtype=float
            ),
            "n_observations": int(
                len(
                    patient_dataframe
                )
            ),
        },
    }

    for level, level_record in level_data.items():
        precomputed = (
            build_calibration_precomputation(
                y_true=level_record[
                    "y_true"
                ],
                probabilities=level_record[
                    "probabilities"
                ],
                n_bins=PRIMARY_ECE_BINS,
            )
        )

        seed_ensemble_precomputations[
            (
                system,
                level,
            )
        ] = precomputed

        metrics = (
            calibration_metrics_from_precomputation(
                precomputed
            )
        )

        metric_row = {
            "system": system,
            "level": level,
            "scope": (
                "three_seed_probability_ensemble"
            ),
            "n_observations": (
                level_record[
                    "n_observations"
                ]
            ),
            "ece_bins": (
                PRIMARY_ECE_BINS
            ),
        }

        metric_row.update(
            metrics
        )

        SEED_ENSEMBLE_METRIC_ROWS.append(
            metric_row
        )

        for class_id in CLASS_IDS:
            SEED_ENSEMBLE_CLASSWISE_ECE_ROWS.append(
                {
                    "system": system,
                    "level": level,
                    "class_id": class_id,
                    "class_name": (
                        CLASS_NAMES[
                            class_id
                        ]
                    ),
                    "classwise_ece": (
                        metrics[
                            f"classwise_ece_{class_id}"
                        ]
                    ),
                    "ece_bins": (
                        PRIMARY_ECE_BINS
                    ),
                }
            )

        SEED_ENSEMBLE_RELIABILITY_ROWS.extend(
            reliability_bin_rows(
                precomputed=precomputed,
                system=system,
                level=level,
                scope=(
                    "three_seed_probability_ensemble"
                ),
                training_seed=np.nan,
            )
        )


seed_ensemble_calibration_df = pd.DataFrame(
    SEED_ENSEMBLE_METRIC_ROWS
)


seed_ensemble_classwise_ece_df = pd.DataFrame(
    SEED_ENSEMBLE_CLASSWISE_ECE_ROWS
)


seed_ensemble_reliability_bins_df = pd.DataFrame(
    SEED_ENSEMBLE_RELIABILITY_ROWS
)


atomic_csv_save(
    seed_ensemble_calibration_df,
    OUTPUT_ROOT
    / "development_seed_ensemble_calibration_metrics.csv",
    index=False,
)


atomic_csv_save(
    seed_ensemble_classwise_ece_df,
    OUTPUT_ROOT
    / "development_seed_ensemble_classwise_ece.csv",
    index=False,
)


atomic_csv_save(
    seed_ensemble_reliability_bins_df,
    SOURCE_DATA_DIR
    / "development_seed_ensemble_reliability_bins.csv",
    index=False,
)


# ==================================================================================================
# 18. ECE bin-count sensitivity
# ==================================================================================================

ece_sensitivity_rows = []


for system in [
    "Swin",
    "ViT",
]:
    for training_seed in TRAINING_SEEDS:
        image_dataframe = selected_sources[
            (
                system,
                training_seed,
            )
        ][
            "dataframe"
        ]

        patient_dataframe = (
            derive_patient_probability_table(
                image_dataframe
            )
        )

        level_data = {
            "image": (
                image_dataframe
            ),
            "patient": (
                patient_dataframe
            ),
        }

        for level, level_dataframe in level_data.items():
            for n_bins in ECE_BIN_SENSITIVITY:
                precomputed = (
                    build_calibration_precomputation(
                        y_true=level_dataframe[
                            "y_true"
                        ].to_numpy(
                            dtype=int
                        ),
                        probabilities=level_dataframe[
                            PROBABILITY_COLUMNS
                        ].to_numpy(
                            dtype=float
                        ),
                        n_bins=n_bins,
                    )
                )

                metrics = (
                    calibration_metrics_from_precomputation(
                        precomputed
                    )
                )

                ece_sensitivity_rows.append(
                    {
                        "scope": (
                            "individual_seed"
                        ),
                        "system": system,
                        "training_seed": (
                            training_seed
                        ),
                        "level": level,
                        "n_bins": n_bins,
                        "top_label_ece": (
                            metrics[
                                "top_label_ece"
                            ]
                        ),
                        "classwise_ece_macro": (
                            metrics[
                                "classwise_ece_macro"
                            ]
                        ),
                    }
                )

    for level in [
        "image",
        "patient",
    ]:
        if level == "image":
            level_dataframe = (
                seed_ensemble_image_dfs[
                    system
                ]
            )

        else:
            level_dataframe = (
                seed_ensemble_patient_dfs[
                    system
                ]
            )

        for n_bins in ECE_BIN_SENSITIVITY:
            precomputed = (
                build_calibration_precomputation(
                    y_true=level_dataframe[
                        "y_true"
                    ].to_numpy(
                        dtype=int
                    ),
                    probabilities=level_dataframe[
                        PROBABILITY_COLUMNS
                    ].to_numpy(
                        dtype=float
                    ),
                    n_bins=n_bins,
                )
            )

            metrics = (
                calibration_metrics_from_precomputation(
                    precomputed
                )
            )

            ece_sensitivity_rows.append(
                {
                    "scope": (
                        "three_seed_probability_ensemble"
                    ),
                    "system": system,
                    "training_seed": np.nan,
                    "level": level,
                    "n_bins": n_bins,
                    "top_label_ece": (
                        metrics[
                            "top_label_ece"
                        ]
                    ),
                    "classwise_ece_macro": (
                        metrics[
                            "classwise_ece_macro"
                        ]
                    ),
                }
            )


ece_bin_sensitivity_df = pd.DataFrame(
    ece_sensitivity_rows
)


atomic_csv_save(
    ece_bin_sensitivity_df,
    OUTPUT_ROOT
    / "development_ece_bin_count_sensitivity.csv",
    index=False,
)


# ==================================================================================================
# 19. Align Swin and ViT seed-ensemble data for paired cluster bootstrap
# ==================================================================================================

swin_image = seed_ensemble_image_dfs[
    "Swin"
].copy()


vit_image = seed_ensemble_image_dfs[
    "ViT"
].copy()


aligned_image = swin_image[
    [
        "image_key",
        "patient_id",
        "y_true",
    ]
    +
    PROBABILITY_COLUMNS
].merge(
    vit_image[
        [
            "image_key",
            "patient_id",
            "y_true",
        ]
        +
        PROBABILITY_COLUMNS
    ],
    on="image_key",
    how="inner",
    suffixes=(
        "_swin",
        "_vit",
    ),
    validate="one_to_one",
)


if (
    len(
        aligned_image
    )
    !=
    EXPECTED_DEVELOPMENT_IMAGES
):
    raise RuntimeError(
        "The seed-ensemble Swin and ViT image files "
        "could not be aligned over all 9,590 images."
    )


if (
    aligned_image[
        "patient_id_swin"
    ]
    !=
    aligned_image[
        "patient_id_vit"
    ]
).any():
    raise RuntimeError(
        "Swin and ViT image patient IDs differ."
    )


if (
    aligned_image[
        "y_true_swin"
    ]
    !=
    aligned_image[
        "y_true_vit"
    ]
).any():
    raise RuntimeError(
        "Swin and ViT image labels differ."
    )


swin_patient = seed_ensemble_patient_dfs[
    "Swin"
].copy()


vit_patient = seed_ensemble_patient_dfs[
    "ViT"
].copy()


aligned_patient = swin_patient[
    [
        "patient_id",
        "y_true",
    ]
    +
    PROBABILITY_COLUMNS
].merge(
    vit_patient[
        [
            "patient_id",
            "y_true",
        ]
        +
        PROBABILITY_COLUMNS
    ],
    on="patient_id",
    how="inner",
    suffixes=(
        "_swin",
        "_vit",
    ),
    validate="one_to_one",
)


if (
    len(
        aligned_patient
    )
    !=
    EXPECTED_DEVELOPMENT_PATIENTS
):
    raise RuntimeError(
        "The seed-ensemble Swin and ViT patient files "
        "could not be aligned over all 478 patients."
    )


if (
    aligned_patient[
        "y_true_swin"
    ]
    !=
    aligned_patient[
        "y_true_vit"
    ]
).any():
    raise RuntimeError(
        "Swin and ViT modal-low patient references differ."
    )


aligned_patient = aligned_patient.sort_values(
    "patient_id"
).reset_index(
    drop=True
)


patient_ids = aligned_patient[
    "patient_id"
].astype(str).to_numpy()


patient_reference = aligned_patient[
    "y_true_swin"
].to_numpy(
    dtype=int
)


patient_index_map = {
    patient_id: index
    for index, patient_id in enumerate(
        patient_ids
    )
}


image_patient_ids = aligned_image[
    "patient_id_swin"
].astype(str).to_numpy()


image_patient_index = np.asarray(
    [
        patient_index_map[
            patient_id
        ]
        for patient_id in image_patient_ids
    ],
    dtype=int,
)


image_y_true = aligned_image[
    "y_true_swin"
].to_numpy(
    dtype=int
)


patient_y_true = patient_reference


bootstrap_precomputations = {
    (
        "Swin",
        "image",
    ): build_calibration_precomputation(
        y_true=image_y_true,
        probabilities=aligned_image[
            [
                "prob_0_swin",
                "prob_1_swin",
                "prob_2_swin",
                "prob_3_swin",
            ]
        ].to_numpy(
            dtype=float
        ),
        n_bins=PRIMARY_ECE_BINS,
    ),
    (
        "ViT",
        "image",
    ): build_calibration_precomputation(
        y_true=image_y_true,
        probabilities=aligned_image[
            [
                "prob_0_vit",
                "prob_1_vit",
                "prob_2_vit",
                "prob_3_vit",
            ]
        ].to_numpy(
            dtype=float
        ),
        n_bins=PRIMARY_ECE_BINS,
    ),
    (
        "Swin",
        "patient",
    ): build_calibration_precomputation(
        y_true=patient_y_true,
        probabilities=aligned_patient[
            [
                "prob_0_swin",
                "prob_1_swin",
                "prob_2_swin",
                "prob_3_swin",
            ]
        ].to_numpy(
            dtype=float
        ),
        n_bins=PRIMARY_ECE_BINS,
    ),
    (
        "ViT",
        "patient",
    ): build_calibration_precomputation(
        y_true=patient_y_true,
        probabilities=aligned_patient[
            [
                "prob_0_vit",
                "prob_1_vit",
                "prob_2_vit",
                "prob_3_vit",
            ]
        ].to_numpy(
            dtype=float
        ),
        n_bins=PRIMARY_ECE_BINS,
    ),
}


# ==================================================================================================
# 20. Generate one shared set of stratified patient-bootstrap samples
# ==================================================================================================

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


patient_indices_by_class = {
    class_id: np.flatnonzero(
        patient_reference
        ==
        class_id
    )
    for class_id in CLASS_IDS
}


bootstrap_patient_multiplicities = np.zeros(
    (
        BOOTSTRAP_REPLICATES,
        EXPECTED_DEVELOPMENT_PATIENTS,
    ),
    dtype=np.int16,
)


for bootstrap_index in range(
    BOOTSTRAP_REPLICATES
):
    multiplicity = np.zeros(
        EXPECTED_DEVELOPMENT_PATIENTS,
        dtype=np.int16,
    )

    for class_id in CLASS_IDS:
        class_indices = (
            patient_indices_by_class[
                class_id
            ]
        )

        sampled_indices = rng.choice(
            class_indices,
            size=len(
                class_indices
            ),
            replace=True,
        )

        multiplicity += np.bincount(
            sampled_indices,
            minlength=(
                EXPECTED_DEVELOPMENT_PATIENTS
            ),
        ).astype(
            np.int16
        )

    bootstrap_patient_multiplicities[
        bootstrap_index
    ] = multiplicity


multiplicity_path = (
    SOURCE_DATA_DIR
    / "bootstrap_patient_multiplicities.npz"
)


np.savez_compressed(
    multiplicity_path,
    patient_ids=patient_ids,
    patient_reference=patient_reference,
    multiplicities=(
        bootstrap_patient_multiplicities
    ),
    seed=BOOTSTRAP_SEED,
)


# ==================================================================================================
# 21. Patient-cluster bootstrap for seed-ensemble calibration metrics
# ==================================================================================================

BOOTSTRAP_METRICS = [
    "nll",
    "multiclass_brier",
    "ranked_probability_score",
    "top_label_ece",
    "classwise_ece_macro",
]


bootstrap_sample_rows = []


print("")
print(
    "Running "
    + str(
        BOOTSTRAP_REPLICATES
    )
    + " stratified patient-cluster bootstrap replicates..."
)


for bootstrap_index in range(
    BOOTSTRAP_REPLICATES
):
    patient_weights = (
        bootstrap_patient_multiplicities[
            bootstrap_index
        ].astype(
            float
        )
    )

    image_weights = patient_weights[
        image_patient_index
    ]

    sample_row = {
        "bootstrap_index": (
            bootstrap_index
        ),
    }

    for system in [
        "Swin",
        "ViT",
    ]:
        image_metrics_bootstrap = (
            calibration_metrics_from_precomputation(
                bootstrap_precomputations[
                    (
                        system,
                        "image",
                    )
                ],
                sample_weights=image_weights,
            )
        )

        patient_metrics_bootstrap = (
            calibration_metrics_from_precomputation(
                bootstrap_precomputations[
                    (
                        system,
                        "patient",
                    )
                ],
                sample_weights=patient_weights,
            )
        )

        for metric_name in BOOTSTRAP_METRICS:
            sample_row[
                system.lower()
                +
                "_image_"
                +
                metric_name
            ] = image_metrics_bootstrap[
                metric_name
            ]

            sample_row[
                system.lower()
                +
                "_patient_"
                +
                metric_name
            ] = patient_metrics_bootstrap[
                metric_name
            ]

    bootstrap_sample_rows.append(
        sample_row
    )

    if (
        bootstrap_index
        +
        1
    ) % 500 == 0:
        print(
            "  completed "
            + str(
                bootstrap_index
                +
                1
            )
            + " / "
            + str(
                BOOTSTRAP_REPLICATES
            )
        )


bootstrap_samples_df = pd.DataFrame(
    bootstrap_sample_rows
)


atomic_csv_save(
    bootstrap_samples_df,
    SOURCE_DATA_DIR
    / "development_calibration_bootstrap_samples.csv",
    index=False,
)


# ==================================================================================================
# 22. Bootstrap confidence intervals
# ==================================================================================================

bootstrap_ci_rows = []


for system in [
    "Swin",
    "ViT",
]:
    for level in [
        "image",
        "patient",
    ]:
        point_row = (
            seed_ensemble_calibration_df[
                (
                    seed_ensemble_calibration_df[
                        "system"
                    ]
                    ==
                    system
                )
                &
                (
                    seed_ensemble_calibration_df[
                        "level"
                    ]
                    ==
                    level
                )
            ]
            .iloc[
                0
            ]
        )

        for metric_name in BOOTSTRAP_METRICS:
            bootstrap_column = (
                system.lower()
                +
                "_"
                +
                level
                +
                "_"
                +
                metric_name
            )

            bootstrap_values = (
                bootstrap_samples_df[
                    bootstrap_column
                ]
                .to_numpy(
                    dtype=float
                )
            )

            valid_values = bootstrap_values[
                np.isfinite(
                    bootstrap_values
                )
            ]

            ci_low, ci_high = np.percentile(
                valid_values,
                [
                    2.5,
                    97.5,
                ],
            )

            bootstrap_ci_rows.append(
                {
                    "system": system,
                    "level": level,
                    "metric": metric_name,
                    "point_estimate": float(
                        point_row[
                            metric_name
                        ]
                    ),
                    "ci_low": float(
                        ci_low
                    ),
                    "ci_high": float(
                        ci_high
                    ),
                    "n_valid_bootstrap": int(
                        len(
                            valid_values
                        )
                    ),
                    "bootstrap_replicates": (
                        BOOTSTRAP_REPLICATES
                    ),
                    "sampling_unit": (
                        "patient cluster"
                    ),
                    "stratification": (
                        "modal_low"
                    ),
                }
            )


bootstrap_ci_df = pd.DataFrame(
    bootstrap_ci_rows
)


atomic_csv_save(
    bootstrap_ci_df,
    OUTPUT_ROOT
    / "development_seed_ensemble_calibration_metrics_with_cluster_bootstrap_ci.csv",
    index=False,
)


# ==================================================================================================
# 23. Paired Swin-minus-ViT bootstrap calibration differences
# ==================================================================================================

paired_difference_rows = []


LOWER_IS_BETTER_METRICS = {
    "nll",
    "multiclass_brier",
    "ranked_probability_score",
    "top_label_ece",
    "classwise_ece_macro",
}


for level in [
    "image",
    "patient",
]:
    for metric_name in BOOTSTRAP_METRICS:
        swin_point = float(
            seed_ensemble_calibration_df[
                (
                    seed_ensemble_calibration_df[
                        "system"
                    ]
                    ==
                    "Swin"
                )
                &
                (
                    seed_ensemble_calibration_df[
                        "level"
                    ]
                    ==
                    level
                )
            ][
                metric_name
            ].iloc[
                0
            ]
        )

        vit_point = float(
            seed_ensemble_calibration_df[
                (
                    seed_ensemble_calibration_df[
                        "system"
                    ]
                    ==
                    "ViT"
                )
                &
                (
                    seed_ensemble_calibration_df[
                        "level"
                    ]
                    ==
                    level
                )
            ][
                metric_name
            ].iloc[
                0
            ]
        )

        swin_bootstrap = bootstrap_samples_df[
            "swin_"
            +
            level
            +
            "_"
            +
            metric_name
        ].to_numpy(
            dtype=float
        )

        vit_bootstrap = bootstrap_samples_df[
            "vit_"
            +
            level
            +
            "_"
            +
            metric_name
        ].to_numpy(
            dtype=float
        )

        bootstrap_difference = (
            swin_bootstrap
            -
            vit_bootstrap
        )

        valid_difference = (
            bootstrap_difference[
                np.isfinite(
                    bootstrap_difference
                )
            ]
        )

        ci_low, ci_high = np.percentile(
            valid_difference,
            [
                2.5,
                97.5,
            ],
        )

        paired_difference_rows.append(
            {
                "level": level,
                "metric": metric_name,
                "swin_point_estimate": (
                    swin_point
                ),
                "vit_point_estimate": (
                    vit_point
                ),
                "swin_minus_vit_difference": (
                    swin_point
                    -
                    vit_point
                ),
                "difference_ci_low": float(
                    ci_low
                ),
                "difference_ci_high": float(
                    ci_high
                ),
                "n_valid_bootstrap": int(
                    len(
                        valid_difference
                    )
                ),
                "direction_note": (
                    "Negative favors Swin"
                    if
                    metric_name
                    in
                    LOWER_IS_BETTER_METRICS
                    else
                    "Positive favors Swin"
                ),
            }
        )


paired_difference_df = pd.DataFrame(
    paired_difference_rows
)


atomic_csv_save(
    paired_difference_df,
    OUTPUT_ROOT
    / "development_seed_ensemble_paired_calibration_differences.csv",
    index=False,
)


# ==================================================================================================
# 24. Publication-ready summary table
# ==================================================================================================

def format_mean_sd(
    mean_value,
    std_value,
):
    return (
        f"{mean_value:.4f} ± "
        f"{std_value:.4f}"
    )


publication_rows = []


for _, row in calibration_summary_df.iterrows():
    publication_rows.append(
        {
            "System": row[
                "system"
            ],
            "Level": (
                "Image"
                if
                row[
                    "level"
                ]
                ==
                "image"
                else
                "Patient"
            ),
            "NLL, mean ± SD": format_mean_sd(
                row[
                    "nll_mean"
                ],
                row[
                    "nll_std"
                ],
            ),
            "Brier, mean ± SD": format_mean_sd(
                row[
                    "multiclass_brier_mean"
                ],
                row[
                    "multiclass_brier_std"
                ],
            ),
            "RPS, mean ± SD": format_mean_sd(
                row[
                    "ranked_probability_score_mean"
                ],
                row[
                    "ranked_probability_score_std"
                ],
            ),
            "Top-label ECE, mean ± SD": (
                format_mean_sd(
                    row[
                        "top_label_ece_mean"
                    ],
                    row[
                        "top_label_ece_std"
                    ],
                )
            ),
            "Classwise ECE, mean ± SD": (
                format_mean_sd(
                    row[
                        "classwise_ece_macro_mean"
                    ],
                    row[
                        "classwise_ece_macro_std"
                    ],
                )
            ),
            "Accuracy, mean ± SD": (
                format_mean_sd(
                    row[
                        "accuracy_mean"
                    ],
                    row[
                        "accuracy_std"
                    ],
                )
            ),
            "Mean confidence, mean ± SD": (
                format_mean_sd(
                    row[
                        "mean_confidence_mean"
                    ],
                    row[
                        "mean_confidence_std"
                    ],
                )
            ),
        }
    )


publication_table_df = pd.DataFrame(
    publication_rows
)


atomic_csv_save(
    publication_table_df,
    OUTPUT_ROOT
    / "table_development_oof_calibration_audit.csv",
    index=False,
)


# ==================================================================================================
# 25. Figure helpers
# ==================================================================================================

def save_figure(
    figure,
    stem,
):
    if SAVE_PNG:
        figure.savefig(
            PNG_DIR
            /
            f"{stem}.png",
            dpi=PNG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )

    if SAVE_SVG:
        figure.savefig(
            SVG_DIR
            /
            f"{stem}.svg",
            bbox_inches="tight",
            facecolor="white",
        )

    if SAVE_PDF:
        figure.savefig(
            PDF_DIR
            /
            f"{stem}.pdf",
            bbox_inches="tight",
            facecolor="white",
        )

    if SHOW_FIGURES:
        plt.show()

    plt.close(
        figure
    )


def system_color(
    system,
):
    return (
        SWIN_COLOR
        if
        system
        ==
        "Swin"
        else
        VIT_COLOR
    )


# ==================================================================================================
# 26. Top-label reliability diagrams
# ==================================================================================================

top_label_figure, top_label_axes = plt.subplots(
    2,
    2,
    figsize=(
        11.5,
        10.0,
    ),
    sharex=True,
    sharey=True,
)


for system_index, system in enumerate(
    [
        "Swin",
        "ViT",
    ]
):
    for level_index, level in enumerate(
        [
            "image",
            "patient",
        ]
    ):
        axis = top_label_axes[
            system_index,
            level_index,
        ]

        bin_dataframe = (
            seed_ensemble_reliability_bins_df[
                (
                    seed_ensemble_reliability_bins_df[
                        "system"
                    ]
                    ==
                    system
                )
                &
                (
                    seed_ensemble_reliability_bins_df[
                        "level"
                    ]
                    ==
                    level
                )
                &
                (
                    seed_ensemble_reliability_bins_df[
                        "calibration_type"
                    ]
                    ==
                    "top_label"
                )
                &
                (
                    seed_ensemble_reliability_bins_df[
                        "n_observations"
                    ]
                    >
                    0
                )
            ]
            .sort_values(
                "bin_id"
            )
        )

        metrics_row = (
            seed_ensemble_calibration_df[
                (
                    seed_ensemble_calibration_df[
                        "system"
                    ]
                    ==
                    system
                )
                &
                (
                    seed_ensemble_calibration_df[
                        "level"
                    ]
                    ==
                    level
                )
            ]
            .iloc[
                0
            ]
        )

        axis.plot(
            [
                0,
                1,
            ],
            [
                0,
                1,
            ],
            linestyle="--",
            color=REFERENCE_COLOR,
            linewidth=1.1,
            label="Perfect calibration",
        )

        counts = bin_dataframe[
            "n_observations"
        ].to_numpy(
            dtype=float
        )

        if len(counts) > 0:
            marker_sizes = (
                35
                +
                180
                *
                counts
                /
                max(
                    counts.max(),
                    1.0,
                )
            )

        else:
            marker_sizes = 35

        axis.plot(
            bin_dataframe[
                "mean_predicted_probability"
            ],
            bin_dataframe[
                "observed_frequency"
            ],
            color=system_color(
                system
            ),
            linewidth=1.5,
            alpha=0.8,
        )

        axis.scatter(
            bin_dataframe[
                "mean_predicted_probability"
            ],
            bin_dataframe[
                "observed_frequency"
            ],
            s=marker_sizes,
            color=system_color(
                system
            ),
            edgecolor="white",
            linewidth=0.7,
            zorder=3,
        )

        level_title = (
            "Image level"
            if
            level
            ==
            "image"
            else
            "Patient level"
        )

        axis.set_title(
            system
            +
            " — "
            +
            level_title
            +
            "\n"
            +
            "ECE="
            +
            f"{metrics_row['top_label_ece']:.3f}"
            +
            ", Brier="
            +
            f"{metrics_row['multiclass_brier']:.3f}"
            +
            ", NLL="
            +
            f"{metrics_row['nll']:.3f}",
            fontweight="bold",
        )

        axis.set_xlim(
            0,
            1,
        )

        axis.set_ylim(
            0,
            1,
        )

        axis.set_aspect(
            "equal",
            adjustable="box",
        )

        axis.set_xlabel(
            "Mean predicted confidence"
        )

        axis.set_ylabel(
            "Observed exact-class accuracy"
        )


top_label_figure.suptitle(
    (
        "Development OOF top-label reliability "
        "of the three-seed probability ensembles"
    ),
    fontsize=14,
    fontweight="bold",
    y=1.01,
)


top_label_figure.text(
    0.5,
    -0.01,
    (
        "Markers represent nonempty equal-width confidence bins; "
        "marker area is proportional to the number of observations. "
        "No calibration model was fitted."
    ),
    ha="center",
    va="top",
    fontsize=9,
    color=NEUTRAL_COLOR,
)


top_label_figure.tight_layout()


save_figure(
    top_label_figure,
    "Figure_calibration_top_label_reliability",
)


# ==================================================================================================
# 27. Patient-level classwise reliability diagrams
# ==================================================================================================

patient_classwise_figure, patient_classwise_axes = plt.subplots(
    2,
    4,
    figsize=(
        16.5,
        8.2,
    ),
    sharex=True,
    sharey=True,
)


for system_index, system in enumerate(
    [
        "Swin",
        "ViT",
    ]
):
    for class_id in CLASS_IDS:
        axis = patient_classwise_axes[
            system_index,
            class_id,
        ]

        bin_dataframe = (
            seed_ensemble_reliability_bins_df[
                (
                    seed_ensemble_reliability_bins_df[
                        "system"
                    ]
                    ==
                    system
                )
                &
                (
                    seed_ensemble_reliability_bins_df[
                        "level"
                    ]
                    ==
                    "patient"
                )
                &
                (
                    seed_ensemble_reliability_bins_df[
                        "calibration_type"
                    ]
                    ==
                    "classwise_one_vs_rest"
                )
                &
                (
                    seed_ensemble_reliability_bins_df[
                        "class_id"
                    ]
                    ==
                    class_id
                )
                &
                (
                    seed_ensemble_reliability_bins_df[
                        "n_observations"
                    ]
                    >
                    0
                )
            ]
            .sort_values(
                "bin_id"
            )
        )

        classwise_row = (
            seed_ensemble_classwise_ece_df[
                (
                    seed_ensemble_classwise_ece_df[
                        "system"
                    ]
                    ==
                    system
                )
                &
                (
                    seed_ensemble_classwise_ece_df[
                        "level"
                    ]
                    ==
                    "patient"
                )
                &
                (
                    seed_ensemble_classwise_ece_df[
                        "class_id"
                    ]
                    ==
                    class_id
                )
            ]
            .iloc[
                0
            ]
        )

        axis.plot(
            [
                0,
                1,
            ],
            [
                0,
                1,
            ],
            linestyle="--",
            color=REFERENCE_COLOR,
            linewidth=1.0,
        )

        counts = bin_dataframe[
            "n_observations"
        ].to_numpy(
            dtype=float
        )

        marker_sizes = (
            30
            +
            150
            *
            counts
            /
            max(
                counts.max(),
                1.0,
            )
        )

        axis.plot(
            bin_dataframe[
                "mean_predicted_probability"
            ],
            bin_dataframe[
                "observed_frequency"
            ],
            color=system_color(
                system
            ),
            linewidth=1.4,
        )

        axis.scatter(
            bin_dataframe[
                "mean_predicted_probability"
            ],
            bin_dataframe[
                "observed_frequency"
            ],
            s=marker_sizes,
            color=system_color(
                system
            ),
            edgecolor="white",
            linewidth=0.6,
            zorder=3,
        )

        axis.set_title(
            system
            +
            " — "
            +
            CLASS_NAMES[
                class_id
            ]
            +
            "\nECE="
            +
            f"{classwise_row['classwise_ece']:.3f}",
            fontweight="bold",
        )

        axis.set_xlim(
            0,
            1,
        )

        axis.set_ylim(
            0,
            1,
        )

        axis.set_aspect(
            "equal",
            adjustable="box",
        )

        axis.set_xlabel(
            "Predicted class probability"
        )

        if class_id == 0:
            axis.set_ylabel(
                "Observed class frequency"
            )


patient_classwise_figure.suptitle(
    (
        "Development OOF patient-level classwise reliability "
        "of the three-seed probability ensembles"
    ),
    fontsize=14,
    fontweight="bold",
    y=1.01,
)


patient_classwise_figure.text(
    0.5,
    -0.01,
    (
        "One-vs-rest reliability is shown for the modal-low "
        "derived patient reference. No calibration parameter "
        "was estimated."
    ),
    ha="center",
    va="top",
    fontsize=9,
    color=NEUTRAL_COLOR,
)


patient_classwise_figure.tight_layout()


save_figure(
    patient_classwise_figure,
    "Figure_calibration_patient_classwise_reliability",
)


# ==================================================================================================
# 28. Image-level classwise reliability diagrams
# ==================================================================================================

image_classwise_figure, image_classwise_axes = plt.subplots(
    2,
    4,
    figsize=(
        16.5,
        8.2,
    ),
    sharex=True,
    sharey=True,
)


for system_index, system in enumerate(
    [
        "Swin",
        "ViT",
    ]
):
    for class_id in CLASS_IDS:
        axis = image_classwise_axes[
            system_index,
            class_id,
        ]

        bin_dataframe = (
            seed_ensemble_reliability_bins_df[
                (
                    seed_ensemble_reliability_bins_df[
                        "system"
                    ]
                    ==
                    system
                )
                &
                (
                    seed_ensemble_reliability_bins_df[
                        "level"
                    ]
                    ==
                    "image"
                )
                &
                (
                    seed_ensemble_reliability_bins_df[
                        "calibration_type"
                    ]
                    ==
                    "classwise_one_vs_rest"
                )
                &
                (
                    seed_ensemble_reliability_bins_df[
                        "class_id"
                    ]
                    ==
                    class_id
                )
                &
                (
                    seed_ensemble_reliability_bins_df[
                        "n_observations"
                    ]
                    >
                    0
                )
            ]
            .sort_values(
                "bin_id"
            )
        )

        classwise_row = (
            seed_ensemble_classwise_ece_df[
                (
                    seed_ensemble_classwise_ece_df[
                        "system"
                    ]
                    ==
                    system
                )
                &
                (
                    seed_ensemble_classwise_ece_df[
                        "level"
                    ]
                    ==
                    "image"
                )
                &
                (
                    seed_ensemble_classwise_ece_df[
                        "class_id"
                    ]
                    ==
                    class_id
                )
            ]
            .iloc[
                0
            ]
        )

        axis.plot(
            [
                0,
                1,
            ],
            [
                0,
                1,
            ],
            linestyle="--",
            color=REFERENCE_COLOR,
            linewidth=1.0,
        )

        counts = bin_dataframe[
            "n_observations"
        ].to_numpy(
            dtype=float
        )

        marker_sizes = (
            30
            +
            150
            *
            counts
            /
            max(
                counts.max(),
                1.0,
            )
        )

        axis.plot(
            bin_dataframe[
                "mean_predicted_probability"
            ],
            bin_dataframe[
                "observed_frequency"
            ],
            color=system_color(
                system
            ),
            linewidth=1.4,
        )

        axis.scatter(
            bin_dataframe[
                "mean_predicted_probability"
            ],
            bin_dataframe[
                "observed_frequency"
            ],
            s=marker_sizes,
            color=system_color(
                system
            ),
            edgecolor="white",
            linewidth=0.6,
            zorder=3,
        )

        axis.set_title(
            system
            +
            " — "
            +
            CLASS_NAMES[
                class_id
            ]
            +
            "\nECE="
            +
            f"{classwise_row['classwise_ece']:.3f}",
            fontweight="bold",
        )

        axis.set_xlim(
            0,
            1,
        )

        axis.set_ylim(
            0,
            1,
        )

        axis.set_aspect(
            "equal",
            adjustable="box",
        )

        axis.set_xlabel(
            "Predicted class probability"
        )

        if class_id == 0:
            axis.set_ylabel(
                "Observed class frequency"
            )


image_classwise_figure.suptitle(
    (
        "Development OOF image-level classwise reliability "
        "of the three-seed probability ensembles"
    ),
    fontsize=14,
    fontweight="bold",
    y=1.01,
)


image_classwise_figure.text(
    0.5,
    -0.01,
    (
        "Image-level curves are descriptive because multiple "
        "images originate from the same patient. "
        "Patient-cluster bootstrap intervals are reported separately."
    ),
    ha="center",
    va="top",
    fontsize=9,
    color=NEUTRAL_COLOR,
)


image_classwise_figure.tight_layout()


save_figure(
    image_classwise_figure,
    "Figure_calibration_image_classwise_reliability",
)


# ==================================================================================================
# 29. Calibration metric comparison across seeds
# ==================================================================================================

metric_figure, metric_axes = plt.subplots(
    2,
    3,
    figsize=(
        14.5,
        8.2,
    )
)


metric_specs = [
    (
        "nll",
        "Negative log-likelihood",
    ),
    (
        "multiclass_brier",
        "Multiclass Brier score",
    ),
    (
        "top_label_ece",
        "Top-label ECE",
    ),
]


for level_index, level in enumerate(
    [
        "image",
        "patient",
    ]
):
    for metric_index, (
        metric_name,
        metric_title,
    ) in enumerate(
        metric_specs
    ):
        axis = metric_axes[
            level_index,
            metric_index,
        ]

        x_positions = {
            "Swin": 0,
            "ViT": 1,
        }

        for system in [
            "Swin",
            "ViT",
        ]:
            values = calibration_by_seed_df[
                (
                    calibration_by_seed_df[
                        "system"
                    ]
                    ==
                    system
                )
                &
                (
                    calibration_by_seed_df[
                        "level"
                    ]
                    ==
                    level
                )
            ][
                metric_name
            ].to_numpy(
                dtype=float
            )

            mean_value = float(
                values.mean()
            )

            std_value = float(
                values.std(
                    ddof=1
                )
            )

            x_value = x_positions[
                system
            ]

            jitter = np.asarray(
                [
                    -0.07,
                    0.00,
                    0.07,
                ]
            )

            axis.scatter(
                x_value
                +
                jitter,
                values,
                color=system_color(
                    system
                ),
                s=40,
                edgecolor="white",
                linewidth=0.6,
                zorder=3,
            )

            axis.errorbar(
                x_value,
                mean_value,
                yerr=std_value,
                fmt="o",
                color=REFERENCE_COLOR,
                markerfacecolor=(
                    system_color(
                        system
                    )
                ),
                markeredgecolor="white",
                markersize=7,
                capsize=5,
                linewidth=1.2,
                zorder=4,
            )

        level_title = (
            "Image level"
            if
            level
            ==
            "image"
            else
            "Patient level"
        )

        axis.set_title(
            level_title
            +
            " — "
            +
            metric_title,
            fontweight="bold",
        )

        axis.set_xticks(
            [
                0,
                1,
            ]
        )

        axis.set_xticklabels(
            [
                "Swin",
                "ViT",
            ]
        )

        axis.set_ylabel(
            "Lower is better"
        )

        axis.set_ylim(
            bottom=0
        )


metric_figure.suptitle(
    (
        "Development OOF calibration metrics across "
        "training seeds"
    ),
    fontsize=14,
    fontweight="bold",
    y=1.01,
)


metric_figure.text(
    0.5,
    -0.01,
    (
        "Colored points are seed-specific values; black-edged "
        "markers and whiskers show mean ± sample SD across "
        "seeds 42, 2025, and 2026."
    ),
    ha="center",
    va="top",
    fontsize=9,
    color=NEUTRAL_COLOR,
)


metric_figure.tight_layout()


save_figure(
    metric_figure,
    "Figure_calibration_metrics_across_seeds",
)


# ==================================================================================================
# 30. Guardrail and interpretation records
# ==================================================================================================

guardrail_payload = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "analysis_plan_fingerprint": (
        ANALYSIS_PLAN_FINGERPRINT
    ),
    "development_only": True,
    "official_partition_accessed": False,
    "model_training_performed": False,
    "checkpoint_loading_performed": False,
    "raw_image_access_performed": False,
    "model_inference_performed": False,
    "calibration_model_fitted": False,
    "temperature_scaling_performed": False,
    "isotonic_regression_performed": False,
    "threshold_optimized": False,
    "model_selection_performed": False,
    "aggregation_reselection_performed": False,
    "patient_reference_reselection_performed": False,
    "primary_analysis_changed": False,
    "interpretation": (
        "Calibration metrics are post hoc development-OOF "
        "diagnostics. They do not alter the selected systems, "
        "locked patient aggregation, derived patient reference, "
        "or official-partition conclusions."
    ),
}


atomic_json_save(
    guardrail_payload,
    OUTPUT_ROOT
    / "NO_CALIBRATION_RESELECTION_GUARDRAIL.json",
)


# ==================================================================================================
# 31. Generated output manifest
# ==================================================================================================

generated_files = sorted(
    [
        path
        for path in OUTPUT_ROOT.rglob(
            "*"
        )
        if (
            path.is_file()
            and
            not path.name.endswith(
                ".temporary"
            )
        )
    ]
)


generated_manifest_rows = []


for path in generated_files:
    generated_manifest_rows.append(
        {
            "relative_path": str(
                path.relative_to(
                    OUTPUT_ROOT
                )
            ),
            "size_bytes": int(
                path.stat().st_size
            ),
            "sha256": file_sha256(
                path
            ),
        }
    )


generated_manifest_df = pd.DataFrame(
    generated_manifest_rows
)


atomic_csv_save(
    generated_manifest_df,
    MANIFEST_DIR
    / "generated_output_manifest.csv",
    index=False,
)


run_manifest = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook": (
        "12_development_oof_probability_calibration_audit.ipynb"
    ),
    "analysis_plan_fingerprint": (
        ANALYSIS_PLAN_FINGERPRINT
    ),
    "input_root": str(
        RA06
    ),
    "output_root": str(
        OUTPUT_ROOT
    ),
    "systems": [
        "Swin",
        "ViT",
    ],
    "seeds": TRAINING_SEEDS,
    "n_images_per_seed": (
        EXPECTED_DEVELOPMENT_IMAGES
    ),
    "n_patients_per_seed": (
        EXPECTED_DEVELOPMENT_PATIENTS
    ),
    "primary_ece_bins": (
        PRIMARY_ECE_BINS
    ),
    "ece_bin_sensitivity": (
        ECE_BIN_SENSITIVITY
    ),
    "bootstrap_replicates": (
        BOOTSTRAP_REPLICATES
    ),
    "bootstrap_seed": (
        BOOTSTRAP_SEED
    ),
    "safety": {
        "training_performed": False,
        "checkpoints_loaded": False,
        "raw_images_accessed": False,
        "model_inference_performed": False,
        "official_partition_accessed": False,
        "calibration_model_fitted": False,
        "threshold_optimized": False,
        "model_selection_performed": False,
        "primary_analysis_changed": False,
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": mpl.__version__,
        "seaborn": sns.__version__,
    },
}


atomic_json_save(
    run_manifest,
    MANIFEST_DIR
    / "calibration_audit_run_manifest.json",
)


# ==================================================================================================
# 32. Print the principal result tables
# ==================================================================================================

print("")
print("=" * 125)
print("PER-SEED DEVELOPMENT OOF CALIBRATION METRICS")
print("=" * 125)

display_columns_by_seed = [
    "system",
    "training_seed",
    "level",
    "nll",
    "multiclass_brier",
    "ranked_probability_score",
    "top_label_ece",
    "classwise_ece_macro",
    "accuracy",
    "mean_confidence",
    "confidence_minus_accuracy",
]


display(
    calibration_by_seed_df[
        display_columns_by_seed
    ].sort_values(
        [
            "level",
            "system",
            "training_seed",
        ]
    )
)


print("")
print("=" * 125)
print("MEAN ± SD ACROSS THREE TRAINING SEEDS")
print("=" * 125)

display_columns_summary = [
    "system",
    "level",
    "nll_mean",
    "nll_std",
    "multiclass_brier_mean",
    "multiclass_brier_std",
    "ranked_probability_score_mean",
    "ranked_probability_score_std",
    "top_label_ece_mean",
    "top_label_ece_std",
    "classwise_ece_macro_mean",
    "classwise_ece_macro_std",
    "accuracy_mean",
    "accuracy_std",
    "mean_confidence_mean",
    "mean_confidence_std",
]


display(
    calibration_summary_df[
        display_columns_summary
    ].sort_values(
        [
            "level",
            "system",
        ]
    )
)


print("")
print("=" * 125)
print("THREE-SEED OOF PROBABILITY-ENSEMBLE CALIBRATION")
print("=" * 125)

display(
    seed_ensemble_calibration_df[
        [
            "system",
            "level",
            "nll",
            "multiclass_brier",
            "ranked_probability_score",
            "top_label_ece",
            "top_label_mce",
            "classwise_ece_macro",
            "accuracy",
            "mean_confidence",
            "confidence_minus_accuracy",
        ]
    ].sort_values(
        [
            "level",
            "system",
        ]
    )
)


print("")
print("=" * 125)
print("SEED-ENSEMBLE CALIBRATION WITH PATIENT-CLUSTER BOOTSTRAP CIs")
print("=" * 125)

display(
    bootstrap_ci_df.sort_values(
        [
            "level",
            "metric",
            "system",
        ]
    )
)


print("")
print("=" * 125)
print("PAIRED SWIN-MINUS-VIT CALIBRATION DIFFERENCES")
print("=" * 125)

display(
    paired_difference_df.sort_values(
        [
            "level",
            "metric",
        ]
    )
)


print("")
print("=" * 125)
print("CLASSWISE ECE OF THE THREE-SEED OOF PROBABILITY ENSEMBLES")
print("=" * 125)

display(
    seed_ensemble_classwise_ece_df.sort_values(
        [
            "level",
            "system",
            "class_id",
        ]
    )
)


# ==================================================================================================
# 33. Final syntax-safe summary
# ==================================================================================================

summary_lines = [
    "12_development_oof_probability_calibration_audit COMPLETED",
    "==========================================================",
    "",
    "Analysis role:",
    (
        "Post hoc development-only probability-calibration audit"
    ),
    "",
    "Systems:",
    "Swin and ViT",
    "",
    "Training seeds:",
    ", ".join(
        str(seed)
        for seed in TRAINING_SEEDS
    ),
    "",
    "Development images per seed:",
    str(
        EXPECTED_DEVELOPMENT_IMAGES
    ),
    "",
    "Development patients per seed:",
    str(
        EXPECTED_DEVELOPMENT_PATIENTS
    ),
    "",
    "Primary ECE bin count:",
    str(
        PRIMARY_ECE_BINS
    ),
    "",
    "ECE bin sensitivity:",
    ", ".join(
        str(value)
        for value in ECE_BIN_SENSITIVITY
    ),
    "",
    "Patient-cluster bootstrap replicates:",
    str(
        BOOTSTRAP_REPLICATES
    ),
    "",
    "Bootstrap seed:",
    str(
        BOOTSTRAP_SEED
    ),
    "",
    "Analysis-plan fingerprint:",
    ANALYSIS_PLAN_FINGERPRINT,
    "",
    "Official partition accessed:",
    "NO",
    "",
    "Model training performed:",
    "NO",
    "",
    "Checkpoint loading performed:",
    "NO",
    "",
    "Model inference performed:",
    "NO",
    "",
    "Calibration model fitted:",
    "NO",
    "",
    "Temperature scaling performed:",
    "NO",
    "",
    "Threshold optimized:",
    "NO",
    "",
    "Model selection performed:",
    "NO",
    "",
    "Primary analysis changed:",
    "NO",
    "",
    "Main calibration table:",
    str(
        OUTPUT_ROOT
        / "table_development_oof_calibration_audit.csv"
    ),
    "",
    "Per-seed metrics:",
    str(
        OUTPUT_ROOT
        / "development_calibration_metrics_by_seed.csv"
    ),
    "",
    "Across-seed summary:",
    str(
        OUTPUT_ROOT
        / "development_calibration_summary_across_seeds.csv"
    ),
    "",
    "Seed-ensemble metrics with bootstrap CIs:",
    str(
        OUTPUT_ROOT
        / "development_seed_ensemble_calibration_metrics_with_cluster_bootstrap_ci.csv"
    ),
    "",
    "Paired calibration differences:",
    str(
        OUTPUT_ROOT
        / "development_seed_ensemble_paired_calibration_differences.csv"
    ),
    "",
    "Reliability-bin source data:",
    str(
        SOURCE_DATA_DIR
        / "development_seed_ensemble_reliability_bins.csv"
    ),
    "",
    "Figures:",
    str(
        FIGURE_ROOT
    ),
    "",
    "Output directory:",
    str(
        OUTPUT_ROOT
    ),
]


summary_text = "\n".join(
    summary_lines
)


atomic_text_save(
    summary_text + "\n",
    OUTPUT_ROOT
    / "CALIBRATION_AUDIT_SUMMARY.txt",
)


print("")
print("=" * 125)
print(
    "12_development_oof_probability_calibration_audit "
    "COMPLETED SUCCESSFULLY"
)
print("=" * 125)

print("")
print("Output directory:")
print(OUTPUT_ROOT)

print("")
print("Analysis-plan fingerprint:")
print(ANALYSIS_PLAN_FINGERPRINT)

print("")
print("Publication-ready calibration table:")
print(
    OUTPUT_ROOT
    / "table_development_oof_calibration_audit.csv"
)

print("")
print("Calibration metrics with bootstrap CIs:")
print(
    OUTPUT_ROOT
    / "development_seed_ensemble_calibration_metrics_with_cluster_bootstrap_ci.csv"
)

print("")
print("Reliability diagrams:")
print(FIGURE_ROOT)

print("")
print("Safety confirmation:")
print(
    "No official-partition access, model training, checkpoint "
    "loading, raw-image access, model inference, calibration "
    "fitting, threshold optimization, or model reselection occurred."
)

In [ ]:
from pathlib import Path
import pandas as pd

src = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/12_development_oof_probability_calibration_audit_v1/development_ece_bin_count_sensitivity.csv')
)

df = pd.read_csv(src)

print(df.to_string(index=False))

txt = src.with_suffix(".txt")
txt.write_text(df.to_csv(index=False), encoding="utf-8")

print("\nSaved:", txt)